<a href="https://colab.research.google.com/github/kvanoost/AGP/blob/main/notebooks/03_teacher_model_selection_P04_DEFAULT_REVISED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 — Teacher-model evaluation and P04 operational teacher registry

This notebook has two clearly separated purposes:

1. **Scientific model evaluation**
   - screen annotation consistency within ecological clusters;
   - compare eight candidate RF teachers;
   - evaluate point-month classification;
   - evaluate complete trajectory behaviour;
   - evaluate annual cropland gains, losses and stable states.

2. **Operational teacher construction**
   - retain all comparative results for interpretation;
   - use **P04 consistently as the default teacher in every cluster**;
   - fit and save one cluster-specific P04 Random Forest for downstream pseudo-label generation.

P04 is **not a CNN**. It is a Random Forest using M02 temporal features plus GEE and NICFI patch-summary features. The later student model may use deep learning after sufficient pseudo-labels have been generated.


## BLOCK 01 — Settings

In [ ]:
# ======================================================================================
# BLOCK 01 — SETTINGS, ROBUST DRIVE MOUNT, PATHS AND PACKAGES
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 01 — SETTINGS, ROBUST DRIVE MOUNT, PATHS AND PACKAGES")
print("=" * 100)

import os, sys, re, json, math, time, warnings, subprocess
from pathlib import Path

def ensure_package(import_name, pip_name=None):
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or import_name]
        )

for import_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]:
    ensure_package(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix,
)
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------
# Robust Colab Drive handling
# ---------------------------
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    candidate_roots = [
        Path("/content/gdrive/MyDrive"),
        Path("/content/drive/MyDrive"),
    ]
    DRIVE_ROOT = next((p for p in candidate_roots if p.exists()), None)

    if DRIVE_ROOT is None:
        mountpoint = Path("/content/gdrive")
        mountpoint.mkdir(parents=True, exist_ok=True)

        if any(mountpoint.iterdir()):
            raise RuntimeError(
                f"{mountpoint} is not a valid mounted Drive and is not empty. "
                "Clean it or set DRIVE_ROOT manually."
            )

        drive.mount(str(mountpoint))
        DRIVE_ROOT = mountpoint / "MyDrive"
else:
    DRIVE_ROOT = Path.home()

# Change only PROJECT_ROOT when reusing the notebook for another project.
PROJECT_ROOT = DRIVE_ROOT / "Colab Notebooks" / "PanAfrica_LU" / "NICFI_phase02"

MODEL_TABLE_DIR = PROJECT_ROOT / "03_model_tables"
CLUSTER_TABLE_DIR = PROJECT_ROOT / "05_ecological_clustering" / "tables"
REFERENCE_PATH = PROJECT_ROOT / "00_inputs" / "congo_phase02_reference.csv"

RUN_ROOT = PROJECT_ROOT / "05_model_runs" / "03_teacher_model_selection"
RUN_ID = time.strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / RUN_ID
TABLE_DIR = RUN_DIR / "tables"
FIGURE_DIR = RUN_DIR / "figures"
MODEL_DIR = RUN_DIR / "models"
CACHE_DIR = RUN_ROOT / "derived_feature_cache"

for directory in [RUN_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------------------------
# Core model-table inputs
# ---------------------------
MODEL_PATHS = {
    "M01_RF_GEE_POINT": MODEL_TABLE_DIR / "model_table_M01_GEE_POINT.csv",
    "M02_RF_GEE_S2_T24": MODEL_TABLE_DIR / "model_table_M02_GEE_S2_T24.csv",
    "M03_RF_GEE_S2_NICFI_T24": MODEL_TABLE_DIR / "model_table_M03_GEE_S2_NICFI_T24.csv",
    "M04_RF_FULL_TABULAR_T12T24": MODEL_TABLE_DIR / "model_table_M04_GEE_S2_NICFI_T12T24.csv",
}

CLUSTER_ASSIGNMENT_CANDIDATES = [
    CLUSTER_TABLE_DIR / "zone_cluster_solutions.csv",
    CLUSTER_TABLE_DIR / "zone_environment_clusters_and_support.csv",
]
CLUSTER_COLUMN = "cluster_k4"

PATCH_INDEX_PATH = MODEL_TABLE_DIR / "patch_index_table_GEE_NICFI.csv"
GEE_PATCH_NPZ = (
    PROJECT_ROOT / "02_features" / "spatial_patches" /
    "gee_embedding_static_labelled" / "gee_embedding_static_labelled_patches.npz"
)
NICFI_PATCH_NPZ = (
    PROJECT_ROOT / "02_features" / "spatial_patches" /
    "nicfi_static_labelled" / "nicfi_static_labelled_patches.npz"
)

RUN_PATCH_MODELS = True
PATCH_CHUNK_SIZE = 512

TARGET_CLASSES = ["built_up", "cropland", "forest", "savannah", "water"]
CROP_CLASS = "cropland"

RANDOM_SEED = 42
N_TREES = 500
N_JOBS = -1

# ---------------------------
# Zone annotation screening
# ---------------------------
MIN_ZONE_ROWS = 40
MIN_ZONE_CROP = 10
MIN_ZONE_NONCROP = 10

AUTO_EXCLUDE_PRECISION_THRESHOLD = 0.60
AUTO_EXCLUDE_RECALL_THRESHOLD = 0.60

# Manual overrides have priority over automatic decisions.
MANUAL_KEEP_ZONES = set()
MANUAL_EXCLUDE_ZONES = set()

# Context only; low BA alone never excludes a zone.
SCREEN_MIN_BA = 0.50

# ---------------------------
# Teacher validation/selection
# ---------------------------
SELECTION_BA_TOLERANCE = 0.01
MIN_TEACHER_TRAIN_ROWS = 100
MIN_TEACHER_TEST_ROWS = 30
MIN_TEACHER_TRAIN_CLASSES = 2
MIN_TEACHER_TEST_CLASSES = 2

# Bootstrap complete unit_id trajectories to estimate metric uncertainty.
# Set to 0 to disable.
N_BOOTSTRAP = 200

# Consistent compact model codes used in all reports and registries.
MODEL_CODES = {
    "M01_RF_GEE_POINT": "M01",
    "M02_RF_GEE_S2_T24": "M02",
    "M03_RF_GEE_S2_NICFI_T24": "M03",
    "M04_RF_FULL_TABULAR_T12T24": "M04",
    "PATCH_GEE_STATS_ONLY": "P01",
    "PATCH_NICFI_STATS_ONLY": "P02",
    "PATCH_GEE_NICFI_STATS": "P03",
    "M02_PLUS_GEE_NICFI_PATCH_STATS": "P04",
}

MODEL_DESCRIPTIONS = {
    "M01_RF_GEE_POINT": "GEE annual point embeddings",
    "M02_RF_GEE_S2_T24": "M01 + S2 T24",
    "M03_RF_GEE_S2_NICFI_T24": "M02 + NICFI T24",
    "M04_RF_FULL_TABULAR_T12T24": "Full tabular T12 + T24",
    "PATCH_GEE_STATS_ONLY": "GEE patch summaries",
    "PATCH_NICFI_STATS_ONLY": "NICFI patch summaries",
    "PATCH_GEE_NICFI_STATS": "GEE + NICFI patch summaries",
    "M02_PLUS_GEE_NICFI_PATCH_STATS": "M02 + GEE/NICFI patch summaries",
}

# --------------------------------------------------------------------------------------
# Operational teacher policy
# --------------------------------------------------------------------------------------
# All candidate models remain evaluated scientifically.
# For downstream pseudo-label generation, use the same architecture in every cluster.
DEFAULT_OPERATIONAL_TEACHER = "M02_PLUS_GEE_NICFI_PATCH_STATS"  # P04

if MODEL_CODES.get(DEFAULT_OPERATIONAL_TEACHER) != "P04":
    raise RuntimeError(
        "DEFAULT_OPERATIONAL_TEACHER must resolve to model code P04."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("CLUSTER_COLUMN:", CLUSTER_COLUMN)
print("RUN_PATCH_MODELS:", RUN_PATCH_MODELS)
print("DEFAULT_OPERATIONAL_TEACHER:", MODEL_CODES[DEFAULT_OPERATIONAL_TEACHER], "—", MODEL_DESCRIPTIONS[DEFAULT_OPERATIONAL_TEACHER])



BLOCK 01 — SETTINGS, ROBUST DRIVE MOUNT, PATHS AND PACKAGES
Mounted at /content/gdrive
PROJECT_ROOT: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02
RUN_DIR: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/05_model_runs/03_teacher_model_selection/run_20260718_203109
CLUSTER_COLUMN: cluster_k4
RUN_PATCH_MODELS: True
DEFAULT_OPERATIONAL_TEACHER: P04 — M02 + GEE/NICFI patch summaries


## BLOCK 02 — Helpers

In [ ]:
# ======================================================================================
# BLOCK 02 — REUSABLE HELPERS, FEATURE CONTROL AND METRICS
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 02 — REUSABLE HELPERS, FEATURE CONTROL AND METRICS")
print("=" * 100)

# Never allow these fields to become predictors.
NON_FEATURE_COLUMNS = {
    # Identifiers and geography
    "row_id", "point_id", "unit_id", "zone_id", "point_month_key",
    "latitude", "longitude",

    # Time metadata
    "target_yearmonth", "target_year", "target_month",
    "observation_year", "observation_month", "observation_yearmonth_raw",

    # Labels and annotation metadata
    "label_main5", "land_cover_main5", "land_cover_consensus",
    "land_cover", "land_cover_norm", "land_cover_raw", "label",
    "label_id", "observation_id", "annotator", "annottr",
    "dominant_annotator", "annotator_short", "annotator_counts",
    "annotator_counts_reference", "n_reference_rows", "n_annotators",

    # Data provenance / role
    "source_file", "source_row", "label_source", "point_type",
    "source_type", "label_role", "target_source_type",
    "is_snapshot_label", "is_trajectory_label",
    "has_snapshot_label", "has_trajectory_label",
    "is_snapshot_point", "is_trajectory_point",
    "is_opportunity_point",

    # Screening/model metadata
    "cluster_id", "confidence",
}

NON_FEATURE_COLUMNS.update({
    "has_any_label",
    "has_both_label_sources",
    "label_conflict_flag",
    "has_snapshot_label",
    "has_trajectory_label",
})

def read_csv_required(path, name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
    df = pd.read_csv(path)
    print(f"Loaded {name}: {path} | shape={df.shape}")
    return df

def find_cluster_assignment_path():
    for path in CLUSTER_ASSIGNMENT_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find a cluster assignment table. Checked:\n - "
        + "\n - ".join(map(str, CLUSTER_ASSIGNMENT_CANDIDATES))
    )

def normalize_core_columns(df):
    """Standardize IDs and create the internal label column land_cover_main5."""
    df = df.copy()

    for column in ["row_id", "point_id", "unit_id", "zone_id", "target_yearmonth"]:
        if column in df.columns:
            df[column] = df[column].astype(str).str.strip()

    label_candidates = [
        "label_main5",
        "land_cover_main5",
        "land_cover_consensus",
        "land_cover",
        "label",
    ]
    label_column = next((c for c in label_candidates if c in df.columns), None)

    if label_column is None:
        possible = [
            c for c in df.columns
            if any(token in c.lower() for token in ["label", "land_cover", "class"])
        ]
        raise RuntimeError(
            "Could not identify the land-cover label column.\n"
            f"Expected one of: {label_candidates}\n"
            f"Possible label-like columns: {possible}"
        )

    df["land_cover_main5"] = (
        df[label_column].astype(str).str.strip().str.lower()
    )
    print(f"Detected label column: {label_column} -> land_cover_main5")
    return df

def derive_label_role(df):
    """Return snapshot / trajectory / unknown for each row."""
    role = pd.Series("unknown", index=df.index, dtype="object")

    for column in ["is_trajectory_label", "has_trajectory_label", "is_trajectory_point"]:
        if column in df.columns:
            values = df[column].astype(str).str.lower().isin(["1", "true", "yes"])
            role.loc[values] = "trajectory"

    for column in ["is_snapshot_label", "has_snapshot_label", "is_snapshot_point"]:
        if column in df.columns:
            values = df[column].astype(str).str.lower().isin(["1", "true", "yes"])
            role.loc[(role == "unknown") & values] = "snapshot"

    text_columns = [
        c for c in ["label_source", "point_type", "source_type"]
        if c in df.columns
    ]
    if text_columns:
        text = (
            df[text_columns].fillna("").astype(str)
            .agg(" ".join, axis=1).str.lower()
        )
        role.loc[text.str.contains("traj", regex=False)] = "trajectory"
        role.loc[
            (role == "unknown") & text.str.contains("snapshot", regex=False)
        ] = "snapshot"

    if "target_source_type" in df.columns:
        values = pd.to_numeric(df["target_source_type"], errors="coerce")
        role.loc[(role == "unknown") & values.eq(1)] = "snapshot"
        role.loc[(role == "unknown") & values.eq(2)] = "trajectory"

    return role

def numeric_predictor_columns(df):
    """Numeric columns after strict removal of all known metadata."""
    output = []
    for column in df.columns:
        if column in NON_FEATURE_COLUMNS:
            continue
        if column.startswith("cluster_"):
            continue
        if pd.api.types.is_numeric_dtype(df[column]):
            output.append(column)
    return output

def gee_embedding_columns(df):
    preferred = [
        c for c in df.columns
        if re.fullmatch(r"geept__A\d+", str(c))
    ]
    if preferred:
        return sorted(preferred)

    fallback = [
        c for c in df.columns
        if re.fullmatch(r"A\d+", str(c))
        and pd.api.types.is_numeric_dtype(df[c])
    ]
    return sorted(fallback)

def patch_columns(df, prefix):
    return sorted([
        c for c in df.columns
        if c.startswith(prefix)
        and pd.api.types.is_numeric_dtype(df[c])
    ])

def model_feature_columns(model_name, df):
    """
    Single authoritative feature selector used by Blocks 07, 08 and 10.

    This prevents metadata leakage and ensures that the feature manifest,
    validation model and final saved teacher use exactly the same columns.
    """
    gee_point = gee_embedding_columns(df)
    gee_patch = patch_columns(df, "geePatch__")
    nicfi_patch = patch_columns(df, "nicfiPatch__")

    if model_name == "M01_RF_GEE_POINT":
        features = gee_point

    elif model_name == "PATCH_GEE_STATS_ONLY":
        features = gee_patch

    elif model_name == "PATCH_NICFI_STATS_ONLY":
        features = nicfi_patch

    elif model_name == "PATCH_GEE_NICFI_STATS":
        features = gee_patch + nicfi_patch

    elif model_name == "M02_PLUS_GEE_NICFI_PATCH_STATS":
        base = [
            c for c in numeric_predictor_columns(df)
            if not c.startswith("geePatch__")
            and not c.startswith("nicfiPatch__")
        ]
        features = base + gee_patch + nicfi_patch

    else:
        # M02, M03 and M04: all valid numeric predictors in their own model table.
        features = numeric_predictor_columns(df)

    features = list(dict.fromkeys(features))

    if not features:
        raise RuntimeError(f"No predictor columns found for {model_name}.")

    leaked = [
        c for c in features
        if c in NON_FEATURE_COLUMNS or c.startswith("cluster_")
    ]
    if leaked:
        raise RuntimeError(
            f"Metadata leakage detected for {model_name}: {leaked}"
        )

    return features

def make_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=N_TREES,
            random_state=RANDOM_SEED,
            n_jobs=N_JOBS,
            class_weight="balanced_subsample",
            min_samples_leaf=2,
            max_features="sqrt",
        )),
    ])

def safe_multiclass_metrics(y_true, y_pred):
    """
    Stable metrics with explicit labels.

    Balanced accuracy is calculated over classes actually present in y_true.
    Suppressing the sklearn warning does not change its numerical value.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    present_true = sorted(set(y_true))
    recalls = recall_score(
        y_true, y_pred,
        labels=present_true,
        average=None,
        zero_division=0,
    )

    crop_true = y_true == CROP_CLASS
    crop_pred = y_pred == CROP_CLASS

    return {
        "n": len(y_true),
        "n_classes_true": len(present_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": float(np.mean(recalls)) if len(recalls) else np.nan,
        "macro_f1": f1_score(
            y_true, y_pred,
            labels=TARGET_CLASSES,
            average="macro",
            zero_division=0,
        ),
        "cropland_n_true": int(crop_true.sum()),
        "cropland_n_pred": int(crop_pred.sum()),
        "cropland_precision": precision_score(
            crop_true, crop_pred, zero_division=0
        ),
        "cropland_recall": recall_score(
            crop_true, crop_pred, zero_division=0
        ),
        "cropland_f1": f1_score(
            crop_true, crop_pred, zero_division=0
        ),
        "cropland_fraction_true": float(crop_true.mean()),
        "cropland_fraction_pred": float(crop_pred.mean()),
    }

def confusion_long(y_true, y_pred, extra=None):
    matrix = confusion_matrix(y_true, y_pred, labels=TARGET_CLASSES)
    rows = []
    for i, truth in enumerate(TARGET_CLASSES):
        for j, prediction in enumerate(TARGET_CLASSES):
            row = {
                "truth": truth,
                "prediction": prediction,
                "n": int(matrix[i, j]),
            }
            if extra:
                row.update(extra)
            rows.append(row)
    return rows

def grouped_bootstrap_metrics(pred_df, n_boot=N_BOOTSTRAP):
    """
    Bootstrap complete unit_id trajectories without model retraining.

    The implementation uses safe_multiclass_metrics(), so rare-class
    bootstrap samples do not emit sklearn warnings.
    """
    if n_boot <= 0 or pred_df["unit_id"].nunique() < 2:
        return {}

    rng = np.random.default_rng(RANDOM_SEED)
    units = pred_df["unit_id"].drop_duplicates().to_numpy()
    groups = {
        unit_id: group
        for unit_id, group in pred_df.groupby("unit_id", sort=False)
    }

    records = []

    for _ in range(n_boot):
        sampled_units = rng.choice(units, size=len(units), replace=True)
        bootstrap = pd.concat(
            [groups[unit_id] for unit_id in sampled_units],
            ignore_index=True,
        )
        records.append(
            safe_multiclass_metrics(
                bootstrap["land_cover_main5"].to_numpy(),
                bootstrap["prediction"].to_numpy(),
            )
        )

    bootstrap_df = pd.DataFrame(records)
    output = {}

    for metric in ["balanced_accuracy", "macro_f1", "cropland_f1"]:
        output[f"{metric}_boot_lo"] = bootstrap_df[metric].quantile(0.025)
        output[f"{metric}_boot_hi"] = bootstrap_df[metric].quantile(0.975)

    return output

print("Helpers ready.")



BLOCK 02 — REUSABLE HELPERS, FEATURE CONTROL AND METRICS
Helpers ready.


## BLOCK 03 — Load clusters and observation roles

In [ ]:
# ======================================================================================
# BLOCK 03 — LOAD CLUSTERS, M01 AND ZONE-LEVEL ANNOTATORS
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 03 — LOAD CLUSTERS, M01 AND ZONE-LEVEL ANNOTATORS")
print("=" * 100)

cluster_path = find_cluster_assignment_path()
cluster_raw = read_csv_required(cluster_path, "cluster assignments")

if "zone_id" not in cluster_raw.columns or CLUSTER_COLUMN not in cluster_raw.columns:
    raise RuntimeError(
        f"Cluster table must contain zone_id and {CLUSTER_COLUMN}. "
        f"Available columns: {list(cluster_raw.columns)}"
    )

cluster_assignments = (
    cluster_raw[["zone_id", CLUSTER_COLUMN]]
    .drop_duplicates("zone_id")
    .rename(columns={CLUSTER_COLUMN: "cluster_id"})
    .copy()
)
cluster_assignments["zone_id"] = cluster_assignments["zone_id"].astype(str).str.strip()
cluster_assignments["cluster_id"] = pd.to_numeric(
    cluster_assignments["cluster_id"], errors="raise"
).astype(int)

m01 = normalize_core_columns(
    read_csv_required(MODEL_PATHS["M01_RF_GEE_POINT"], "M01 model table")
)
m01["label_role"] = derive_label_role(m01)
m01 = m01[m01["land_cover_main5"].isin(TARGET_CLASSES)].copy()
m01 = m01.merge(
    cluster_assignments,
    on="zone_id",
    how="left",
    validate="many_to_one",
)

# ---------------------------
# Zone-level annotator lookup
# ---------------------------
reference = read_csv_required(
    REFERENCE_PATH,
    "Phase02 reference observations",
)

if "zone_id" not in reference.columns:
    raise RuntimeError(
        f"Reference file has no zone_id. Available: {list(reference.columns)}"
    )

annotator_candidates = [
    c for c in reference.columns
    if "annot" in c.lower()
]
if not annotator_candidates:
    raise RuntimeError(
        f"No annotator-like column found. Available: {list(reference.columns)}"
    )

annotator_column = (
    "annottr" if "annottr" in reference.columns
    else annotator_candidates[0]
)

reference["zone_id"] = reference["zone_id"].astype(str).str.strip()
reference[annotator_column] = (
    reference[annotator_column]
    .fillna("unknown")
    .astype(str)
    .str.strip()
)

zone_annotator_lookup = (
    reference.groupby("zone_id", as_index=False)
    .agg(
        dominant_annotator=(
            annotator_column,
            lambda x: x.value_counts().index[0]
            if len(x.value_counts()) else "unknown",
        ),
        annotator_counts=(
            annotator_column,
            lambda x: "; ".join(
                f"{name}:{count}"
                for name, count in x.value_counts().items()
            ),
        ),
        n_reference_rows=(annotator_column, "size"),
        n_annotators=(annotator_column, "nunique"),
    )
)

m01 = m01.merge(
    zone_annotator_lookup,
    on="zone_id",
    how="left",
    validate="many_to_one",
)

m01["dominant_annotator"] = m01["dominant_annotator"].fillna("unknown")
m01["annotator_counts"] = m01["annotator_counts"].fillna("unknown")

missing_cluster = sorted(
    m01.loc[m01["cluster_id"].isna(), "zone_id"].dropna().unique()
)
if missing_cluster:
    print("Rows from zones without cluster assignment will be ignored:", missing_cluster)

m01_clustered = m01[m01["cluster_id"].notna()].copy()
m01_clustered["cluster_id"] = m01_clustered["cluster_id"].astype(int)

snapshot_m01 = m01_clustered[
    m01_clustered["label_role"].eq("snapshot")
].copy()
trajectory_m01 = m01_clustered[
    m01_clustered["label_role"].eq("trajectory")
].copy()

if snapshot_m01.empty:
    raise RuntimeError("No snapshot rows detected.")
if trajectory_m01.empty:
    raise RuntimeError("No trajectory rows detected.")

gee_cols = model_feature_columns("M01_RF_GEE_POINT", snapshot_m01)

role_counts = (
    m01_clustered.groupby(["cluster_id", "label_role"], as_index=False)
    .agg(
        n_rows=("row_id", "size"),
        n_units=("unit_id", "nunique"),
        n_zones=("zone_id", "nunique"),
    )
)
display(role_counts)

print("\nSnapshot rows:", len(snapshot_m01))
print("Trajectory rows:", len(trajectory_m01))
print("GEE embedding features:", len(gee_cols))
print("Clusters:", sorted(snapshot_m01["cluster_id"].unique()))
print(
    "Snapshot rows with unknown annotator:",
    int(snapshot_m01["dominant_annotator"].eq("unknown").sum()),
)
print(
    "Trajectory rows with unknown annotator:",
    int(trajectory_m01["dominant_annotator"].eq("unknown").sum()),
)



BLOCK 03 — LOAD CLUSTERS, M01 AND ZONE-LEVEL ANNOTATORS
Loaded cluster assignments: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/05_ecological_clustering/tables/zone_cluster_solutions.csv | shape=(34, 4)
Loaded M01 model table: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/03_model_tables/model_table_M01_GEE_POINT.csv | shape=(10312, 82)
Detected label column: label_main5 -> land_cover_main5
Loaded Phase02 reference observations: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/00_inputs/congo_phase02_reference.csv | shape=(9737, 10)
Rows from zones without cluster assignment will be ignored: ['trajectory_opportunity']


,cluster_id,label_role,n_rows,n_units,n_zones
0,1,snapshot,646,646,4
1,1,trajectory,303,63,4
2,2,snapshot,1842,1842,10
3,2,trajectory,951,213,10
4,3,snapshot,808,808,5
5,3,trajectory,353,101,5
6,4,snapshot,4066,4066,15
7,4,trajectory,1312,278,15



Snapshot rows: 7362
Trajectory rows: 2919
GEE embedding features: 64
Clusters: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Snapshot rows with unknown annotator: 0
Trajectory rows with unknown annotator: 0


## BLOCK 04 — Annotation screening

Each snapshot zone is predicted by an M01 RF trained on **other snapshot zones in the same cluster**. This is a diagnostic of annotation–feature agreement, not proof of annotation error. The next block applies conservative exclusion rules and records the exact reason.


In [ ]:
# ======================================================================================
# BLOCK 04 — SAME-CLUSTER LEAVE-ONE-ZONE-OUT ANNOTATION SCREEN
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 04 — SAME-CLUSTER LEAVE-ONE-ZONE-OUT ANNOTATION SCREEN")
print("=" * 100)

screen_predictions = []
screen_metrics = []
screen_confusion = []

for cluster_id in sorted(snapshot_m01["cluster_id"].unique()):
    cluster_df = snapshot_m01[
        snapshot_m01["cluster_id"].eq(cluster_id)
    ].copy()
    zones = sorted(cluster_df["zone_id"].dropna().unique())

    print("\n" + "-" * 100)
    print(
        f"CLUSTER {cluster_id} | snapshot rows={len(cluster_df)} | zones={len(zones)}"
    )
    print("-" * 100)

    for zone_id in zones:
        test = cluster_df[cluster_df["zone_id"].eq(zone_id)].copy()
        train = cluster_df[~cluster_df["zone_id"].eq(zone_id)].copy()

        test_units = set(test["unit_id"].dropna().astype(str))
        train = train[
            ~train["unit_id"].astype(str).isin(test_units)
        ].copy()

        dominant_annotator = (
            test["dominant_annotator"].dropna().astype(str).iloc[0]
            if test["dominant_annotator"].notna().any()
            else "unknown"
        )
        annotator_counts = (
            test["annotator_counts"].dropna().astype(str).iloc[0]
            if test["annotator_counts"].notna().any()
            else "unknown"
        )

        n_test = len(test)
        n_train = len(train)
        n_crop_test = int(test["land_cover_main5"].eq(CROP_CLASS).sum())
        n_noncrop_test = int(test["land_cover_main5"].ne(CROP_CLASS).sum())
        n_train_classes = int(train["land_cover_main5"].nunique())
        n_test_classes = int(test["land_cover_main5"].nunique())

        status = "evaluated"
        skip_reason = ""

        if n_test < MIN_ZONE_ROWS:
            status, skip_reason = "insufficient_support", f"n_rows<{MIN_ZONE_ROWS}"
        elif n_crop_test < MIN_ZONE_CROP:
            status, skip_reason = "insufficient_support", f"cropland_n<{MIN_ZONE_CROP}"
        elif n_noncrop_test < MIN_ZONE_NONCROP:
            status, skip_reason = "insufficient_support", f"noncropland_n<{MIN_ZONE_NONCROP}"
        elif n_train < MIN_TEACHER_TRAIN_ROWS:
            status, skip_reason = "insufficient_training", "too_few_other_cluster_rows"
        elif n_train_classes < MIN_TEACHER_TRAIN_CLASSES:
            status, skip_reason = "insufficient_training", "too_few_training_classes"

        base = {
            "cluster_id": cluster_id,
            "zone_id": zone_id,
            "dominant_annotator": dominant_annotator,
            "annotator_counts_reference": annotator_counts,
            "screen_status": status,
            "skip_reason": skip_reason,
            "n_train": n_train,
            "n_test": n_test,
            "n_train_zones": int(train["zone_id"].nunique()),
            "n_train_classes": n_train_classes,
            "n_test_classes": n_test_classes,
            "cropland_n_true": n_crop_test,
            "noncropland_n_true": n_noncrop_test,
        }

        if status != "evaluated":
            screen_metrics.append(base)
            print(
                f"{zone_id} | annotator={dominant_annotator} | "
                f"{status}: {skip_reason}"
            )
            continue

        model = make_rf()
        sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=train["land_cover_main5"],
        )
        model.fit(
            train[gee_cols],
            train["land_cover_main5"],
            rf__sample_weight=sample_weight,
        )

        predictions = model.predict(test[gee_cols])
        probabilities = model.predict_proba(test[gee_cols])
        classes = list(model.named_steps["rf"].classes_)

        pred_columns = [
            c for c in [
                "row_id", "unit_id", "zone_id", "target_yearmonth",
                "cluster_id", "land_cover_main5", "dominant_annotator",
            ]
            if c in test.columns
        ]
        pred_df = test[pred_columns].copy()
        pred_df["prediction"] = predictions
        pred_df["prediction_correct"] = (
            pred_df["prediction"].eq(pred_df["land_cover_main5"])
        )
        pred_df["max_probability"] = probabilities.max(axis=1)

        for class_index, class_name in enumerate(classes):
            pred_df[f"p_{class_name}"] = probabilities[:, class_index]

        pred_df["screen_model"] = "M01_same_cluster_leave_one_zone_out"
        screen_predictions.append(pred_df)

        metrics = safe_multiclass_metrics(
            test["land_cover_main5"].to_numpy(),
            predictions,
        )

        precision = metrics["cropland_precision"]
        recall = metrics["cropland_recall"]

        if precision >= 0.60 and recall >= 0.60:
            crop_diagnostic = "consistent_cropland"
        elif recall >= 0.60 and precision < 0.60:
            crop_diagnostic = "high_recall_low_precision"
        elif precision >= 0.60 and recall < 0.60:
            crop_diagnostic = "high_precision_low_recall"
        else:
            crop_diagnostic = "two_way_cropland_disagreement"

        metrics.update(base)
        metrics.update({
            "screen_status": "evaluated",
            "skip_reason": "",
            "crop_diagnostic": crop_diagnostic,
            "n_train_units": int(train["unit_id"].nunique()),
            "n_test_units": int(test["unit_id"].nunique()),
        })
        screen_metrics.append(metrics)

        screen_confusion.extend(
            confusion_long(
                test["land_cover_main5"].to_numpy(),
                predictions,
                {
                    "cluster_id": cluster_id,
                    "zone_id": zone_id,
                    "dominant_annotator": dominant_annotator,
                    "stage": "zone_annotation_screen",
                },
            )
        )

        print(
            f"{zone_id} | annotator={dominant_annotator} | "
            f"BA={metrics['balanced_accuracy']:.3f} | "
            f"crop precision={precision:.3f} | "
            f"crop recall={recall:.3f} | "
            f"crop F1={metrics['cropland_f1']:.3f}"
        )

screen_metrics_df = pd.DataFrame(screen_metrics)
screen_predictions_df = (
    pd.concat(screen_predictions, ignore_index=True)
    if screen_predictions else pd.DataFrame()
)
screen_confusion_df = pd.DataFrame(screen_confusion)

display_columns = [
    c for c in [
        "cluster_id", "zone_id", "dominant_annotator",
        "n_test", "cropland_n_true", "balanced_accuracy",
        "cropland_precision", "cropland_recall", "cropland_f1",
        "crop_diagnostic", "screen_status", "skip_reason",
    ]
    if c in screen_metrics_df.columns
]

display(
    screen_metrics_df[display_columns]
    .sort_values(
        ["cluster_id", "screen_status", "cropland_f1"],
        na_position="last",
    )
)



BLOCK 04 — SAME-CLUSTER LEAVE-ONE-ZONE-OUT ANNOTATION SCREEN

----------------------------------------------------------------------------------------------------
CLUSTER 1 | snapshot rows=646 | zones=4
----------------------------------------------------------------------------------------------------
G2001_Z096_dynamic_high_change | annotator=group20_Pengzhi | BA=0.410 | crop precision=0.679 | crop recall=0.936 | crop F1=0.787
G2003_Z098_dynamic_high_change | annotator=group20_Pengzhi | BA=0.416 | crop precision=0.452 | crop recall=0.778 | crop F1=0.571
G6002_Z297_stable_low_change | annotator=group60_kristof | BA=0.478 | crop precision=0.636 | crop recall=0.389 | crop F1=0.483
G6005_Z300_stable_low_change | annotator=group60_kristof | insufficient_support: cropland_n<10

----------------------------------------------------------------------------------------------------
CLUSTER 2 | snapshot rows=1842 | zones=10
-----------------------------------------------------------------------

,cluster_id,zone_id,dominant_annotator,n_test,cropland_n_true,balanced_accuracy,cropland_precision,cropland_recall,cropland_f1,crop_diagnostic,screen_status,skip_reason
2,1,G6002_Z297_stable_low_change,group60_kristof,52,18,0.477778,0.636364,0.388889,0.482759,high_precision_low_recall,evaluated,
1,1,G2003_Z098_dynamic_high_change,group20_Pengzhi,138,36,0.416100,0.451613,0.777778,0.571429,high_recall_low_precision,evaluated,
0,1,G2001_Z096_dynamic_high_change,group20_Pengzhi,412,140,0.410345,0.678756,0.935714,0.786787,consistent_cropland,evaluated,
3,1,G6005_Z300_stable_low_change,group60_kristof,44,4,NaN,NaN,NaN,NaN,NaN,insufficient_support,cropland_n<10
6,2,G1002_Z047_dynamic_high_change,group10_Xiaojing,96,31,0.522340,0.455882,1.000000,0.626263,high_recall_low_precision,evaluated,
10,2,G6003_Z298_stable_low_change,group60_kristof,136,59,0.752475,0.847826,0.661017,0.742857,consistent_cropland,evaluated,
5,2,G0204_Z009_dynamic_high_change,group02_kristof,40,12,0.481250,0.600000,1.000000,0.750000,consistent_cropland,evaluated,
11,2,G6004_Z299_stable_low_change,group60_kristof,120,60,0.647053,0.886792,0.783333,0.831858,consistent_cropland,evaluated,
7,2,G1004_Z049_dynamic_high_change,group10_Xiaojing,76,24,0.680976,0.750000,1.000000,0.857143,consistent_cropland,evaluated,
13,2,Tshikapa,Tshikapa,255,108,0.875512,0.808000,0.935185,0.866953,consistent_cropland,evaluated,


## BLOCK 05 — Decide and report discarded zones

In [ ]:
# ======================================================================================
# BLOCK 05 — AUTOMATIC CLUSTER-CONSISTENCY EXCLUSIONS AND COMPACT REPORT
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 05 — AUTOMATIC CLUSTER-CONSISTENCY EXCLUSIONS AND COMPACT REPORT")
print("=" * 100)

zone_decisions = screen_metrics_df.copy()

evaluated_rows = zone_decisions[
    zone_decisions["screen_status"].eq("evaluated")
].copy()

cluster_reference = (
    evaluated_rows.groupby("cluster_id", as_index=False)
    .agg(
        cluster_median_crop_precision=("cropland_precision", "median"),
        cluster_median_crop_recall=("cropland_recall", "median"),
        cluster_median_crop_f1=("cropland_f1", "median"),
        n_evaluated_zones=("zone_id", "nunique"),
    )
)

zone_decisions = zone_decisions.merge(
    cluster_reference,
    on="cluster_id",
    how="left",
)

zone_decisions["crop_fraction_bias"] = (
    zone_decisions["cropland_fraction_pred"]
    - zone_decisions["cropland_fraction_true"]
)

def shorten_annotator_name(value):
    if pd.isna(value):
        return "unknown"

    text = str(value).strip()
    if not text or text.lower() == "unknown":
        return "unknown"

    text = re.sub(r"^group\d+[_\-\s]*", "", text, flags=re.IGNORECASE)
    parts = [part for part in re.split(r"[_\-\s]+", text) if part]
    ignored = {
        "new", "old", "revised", "revision", "final",
        "data", "annotations", "annotation",
    }
    useful = [part for part in parts if part.lower() not in ignored]
    return (useful or parts or [str(value)])[0].capitalize()

zone_decisions["annotator_short"] = (
    zone_decisions["dominant_annotator"]
    .apply(shorten_annotator_name)
)

def classify_zone(row):
    zone_id = row["zone_id"]

    if zone_id in MANUAL_KEEP_ZONES:
        return pd.Series({
            "discard_zone": False,
            "final_status": "KEEP_MANUAL",
            "decision_source": "manual_keep",
            "decision_reason": "Manually forced to keep.",
        })

    if zone_id in MANUAL_EXCLUDE_ZONES:
        return pd.Series({
            "discard_zone": True,
            "final_status": "EXCLUDE_MANUAL",
            "decision_source": "manual_exclude",
            "decision_reason": "Manually forced to exclude.",
        })

    if row.get("screen_status") != "evaluated":
        return pd.Series({
            "discard_zone": False,
            "final_status": "KEEP_INSUFFICIENT_SUPPORT",
            "decision_source": "automatic_keep_insufficient_evidence",
            "decision_reason": (
                "Screen inconclusive: "
                f"{row.get('skip_reason', 'unknown reason')}."
            ),
        })

    precision = row["cropland_precision"]
    recall = row["cropland_recall"]
    crop_f1 = row["cropland_f1"]
    ba = row["balanced_accuracy"]

    if (
        precision < AUTO_EXCLUDE_PRECISION_THRESHOLD
        and recall < AUTO_EXCLUDE_RECALL_THRESHOLD
    ):
        return pd.Series({
            "discard_zone": True,
            "final_status": "EXCLUDE_LOW_CROP_PRECISION_RECALL",
            "decision_source": "automatic_cluster_consistency_rule",
            "decision_reason": (
                f"Cropland precision={precision:.3f}<"
                f"{AUTO_EXCLUDE_PRECISION_THRESHOLD:.2f} and "
                f"recall={recall:.3f}<"
                f"{AUTO_EXCLUDE_RECALL_THRESHOLD:.2f}; "
                f"F1={crop_f1:.3f}, BA={ba:.3f}."
            ),
        })

    if (
        precision >= AUTO_EXCLUDE_PRECISION_THRESHOLD
        and recall >= AUTO_EXCLUDE_RECALL_THRESHOLD
    ):
        status = "KEEP_CONSISTENT"
        reason = (
            f"Cropland precision={precision:.3f} and recall={recall:.3f} "
            "both meet the cluster-consistency threshold."
        )
    elif recall >= AUTO_EXCLUDE_RECALL_THRESHOLD:
        status = "REVIEW_HIGH_RECALL_LOW_PRECISION"
        reason = (
            f"Recall={recall:.3f} is acceptable, but "
            f"precision={precision:.3f} is low."
        )
    else:
        status = "REVIEW_HIGH_PRECISION_LOW_RECALL"
        reason = (
            f"Precision={precision:.3f} is acceptable, but "
            f"recall={recall:.3f} is low."
        )

    if pd.notna(ba) and ba < SCREEN_MIN_BA:
        reason += f" Overall five-class BA={ba:.3f} is low but is contextual only."

    return pd.Series({
        "discard_zone": False,
        "final_status": status,
        "decision_source": "automatic_diagnostic",
        "decision_reason": reason,
    })

decision_columns = zone_decisions.apply(classify_zone, axis=1)
zone_decisions = pd.concat([zone_decisions, decision_columns], axis=1)

discarded_zones = sorted(
    zone_decisions.loc[zone_decisions["discard_zone"], "zone_id"]
    .astype(str).tolist()
)
retained_zones = sorted(
    zone_decisions.loc[~zone_decisions["discard_zone"], "zone_id"]
    .astype(str).tolist()
)
review_zones = sorted(
    zone_decisions.loc[
        zone_decisions["final_status"].astype(str).str.startswith("REVIEW"),
        "zone_id",
    ].astype(str).tolist()
)

print("\n" + "-" * 80)
print("EXCLUDED ZONES")
print("-" * 80)

if discarded_zones:
    display(
        zone_decisions.loc[
            zone_decisions["discard_zone"],
            [
                "cluster_id", "zone_id", "dominant_annotator",
                "n_test", "cropland_n_true",
                "cropland_precision", "cropland_recall",
                "cropland_f1", "decision_reason",
            ],
        ].sort_values(["cluster_id", "cropland_f1"])
    )
else:
    print("None.")

compact = zone_decisions.copy()
compact["zone_short"] = (
    compact["zone_id"].astype(str)
    .str.replace("_dynamic_high_change", "", regex=False)
    .str.replace("_stable_low_change", "", regex=False)
)

status_map = {
    "EXCLUDE_LOW_CROP_PRECISION_RECALL": "EXCLUDE",
    "EXCLUDE_MANUAL": "EXCLUDE",
    "REVIEW_HIGH_RECALL_LOW_PRECISION": "EXTRA_CROP",
    "REVIEW_HIGH_PRECISION_LOW_RECALL": "MISSED_CROP",
    "KEEP_INSUFFICIENT_SUPPORT": "LOW_N",
    "KEEP_CONSISTENT": "KEEP",
    "KEEP_MANUAL": "KEEP_M",
}

compact["Call"] = compact["final_status"].map(status_map).fillna(compact["final_status"])

compact = compact[
    [
        "cluster_id", "zone_short", "annotator_short",
        "n_test", "cropland_n_true",
        "cropland_precision", "cropland_recall",
        "cropland_f1", "crop_fraction_bias", "Call",
    ]
].copy()

compact.columns = [
    "Cl", "Zone", "Annot", "N", "CropN",
    "Prec", "Rec", "F1", "Bias", "Call",
]

for column in ["Prec", "Rec", "F1", "Bias"]:
    compact[column] = compact[column].round(2)

call_order = {
    "EXCLUDE": 1,
    "EXTRA_CROP": 2,
    "MISSED_CROP": 3,
    "LOW_N": 4,
    "KEEP": 5,
    "KEEP_M": 6,
}
compact["_order"] = compact["Call"].map(call_order).fillna(99)
compact = (
    compact.sort_values(["_order", "Cl", "F1"], na_position="last")
    .drop(columns="_order")
)

print("\n" + "-" * 80)
print("ZONE DECISION TABLE")
print("-" * 80)
display(compact)

zone_decisions.to_csv(
    TABLE_DIR / "zone_annotation_screening_decisions.csv",
    index=False,
)
compact.to_csv(
    TABLE_DIR / "zone_annotation_decision_table_small.csv",
    index=False,
)
screen_metrics_df.to_csv(
    TABLE_DIR / "zone_annotation_screening_metrics.csv",
    index=False,
)
screen_predictions_df.to_csv(
    TABLE_DIR / "zone_annotation_screening_predictions.csv",
    index=False,
)
screen_confusion_df.to_csv(
    TABLE_DIR / "zone_annotation_screening_confusion.csv",
    index=False,
)

for filename, values in [
    ("discarded_zones.json", discarded_zones),
    ("retained_zones.json", retained_zones),
    ("review_zones.json", review_zones),
]:
    with open(TABLE_DIR / filename, "w") as file:
        json.dump(values, file, indent=2)

print("\n" + "=" * 100)
print("BLOCK 05 SUMMARY")
print("=" * 100)
print("Retained zones:", len(retained_zones))
print("Review zones:", len(review_zones))
print("Excluded zones:", len(discarded_zones))
print("Excluded zone IDs:", discarded_zones)
print(
    "Automatic exclusion: cropland precision and recall are both below "
    f"{AUTO_EXCLUDE_PRECISION_THRESHOLD:.2f}."
)



BLOCK 05 — AUTOMATIC CLUSTER-CONSISTENCY EXCLUSIONS AND COMPACT REPORT

--------------------------------------------------------------------------------
EXCLUDED ZONES
--------------------------------------------------------------------------------


,cluster_id,zone_id,dominant_annotator,n_test,cropland_n_true,cropland_precision,cropland_recall,cropland_f1,decision_reason
31,4,G3005_Z150_dynamic_high_change,group30_Nadine,43,10,0.571429,0.400000,0.470588,Cropland precision=0.571<0.60 and recall=0.400...
28,4,G3002_Z147_dynamic_high_change,group30_Nadine,111,24,0.482759,0.583333,0.528302,Cropland precision=0.483<0.60 and recall=0.583...
27,4,G3001_Z146_dynamic_high_change,group30_Nadine,195,38,0.552632,0.552632,0.552632,Cropland precision=0.553<0.60 and recall=0.553...



--------------------------------------------------------------------------------
ZONE DECISION TABLE
--------------------------------------------------------------------------------


,Cl,Zone,Annot,N,CropN,Prec,Rec,F1,Bias,Call
31,4,G3005_Z150,Nadine,43,10,0.57,0.40,0.47,-0.07,EXCLUDE
28,4,G3002_Z147,Nadine,111,24,0.48,0.58,0.53,0.05,EXCLUDE
27,4,G3001_Z146,Nadine,195,38,0.55,0.55,0.55,0.00,EXCLUDE
1,1,G2003_Z098,Pengzhi,138,36,0.45,0.78,0.57,0.19,EXTRA_CROP
6,2,G1002_Z047,Xiaojing,96,31,0.46,1.00,0.63,0.39,EXTRA_CROP
14,3,G0102_Z002,Yanfei,405,107,0.34,0.94,0.50,0.47,EXTRA_CROP
23,4,G1003_Z048,Xiaojing,103,12,0.34,0.92,0.50,0.19,EXTRA_CROP
2,1,G6002_Z297,Kristof,52,18,0.64,0.39,0.48,-0.13,MISSED_CROP
18,3,G0205_Z010,Kristof,104,48,0.68,0.52,0.59,-0.11,MISSED_CROP
26,4,G2005_Z100,Pengzhi,377,130,0.88,0.58,0.70,-0.12,MISSED_CROP



BLOCK 05 SUMMARY
Retained zones: 31
Review zones: 7
Excluded zones: 3
Excluded zone IDs: ['G3001_Z146_dynamic_high_change', 'G3002_Z147_dynamic_high_change', 'G3005_Z150_dynamic_high_change']
Automatic exclusion: cropland precision and recall are both below 0.60.


## BLOCK 06 — Patch summaries

This block is optional. It builds one reusable summary-feature cache from the existing GEE and NICFI patch NPZ files. It does **not** create new Earth Engine features. Set `RUN_PATCH_MODELS=False` in Block 01 to run only M01–M04.


In [ ]:
# ======================================================================================
# BLOCK 06 — REUSABLE PATCH-SUMMARY CACHE
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 06 — REUSABLE PATCH-SUMMARY CACHE")
print("=" * 100)

PATCH_SUMMARY_CACHE = CACHE_DIR / "patch_summary_features_all_labelled_rows.csv"
PATCH_CACHE_MANIFEST = CACHE_DIR / "patch_summary_features_manifest.json"

def largest_numeric_4d_array(npz_object):
    candidates = []

    for key in npz_object.files:
        array = npz_object[key]

        if array.ndim != 4 or array.dtype == object:
            continue

        candidates.append((array.size, key, array))

    if not candidates:
        raise RuntimeError(
            "No numeric 4-D patch array found. "
            f"Available keys: {npz_object.files}"
        )

    candidates.sort(key=lambda item: item[0], reverse=True)
    _, key, array = candidates[0]

    print(
        f"Selected patch array: {key} | "
        f"shape={array.shape} | dtype={array.dtype}"
    )
    return array

def find_index_column(index_df, sensor):
    candidates = [
        c for c in index_df.columns
        if sensor.lower() in c.lower()
        and "index" in c.lower()
    ]
    return candidates[0] if candidates else None

def summarize_patch_array(patches, prefix, chunk_size=PATCH_CHUNK_SIZE):
    """
    Center, mean, SD and p10/p50/p90 per band plus valid fraction.
    Percentiles are calculated in one call for efficiency.
    """
    n, height, width, bands = patches.shape
    rows = []
    center_y, center_x = height // 2, width // 2

    for start in range(0, n, chunk_size):
        stop = min(start + chunk_size, n)
        chunk = np.asarray(patches[start:stop], dtype=np.float32)
        flat = chunk.reshape(chunk.shape[0], -1, bands)

        center = chunk[:, center_y, center_x, :]
        mean = np.nanmean(flat, axis=1)
        std = np.nanstd(flat, axis=1)
        percentiles = np.nanpercentile(flat, [10, 50, 90], axis=1)
        p10, p50, p90 = percentiles
        valid_fraction = np.isfinite(flat).mean(axis=(1, 2))

        rows.append(
            np.concatenate(
                [
                    center, mean, std, p10, p50, p90,
                    valid_fraction[:, None],
                ],
                axis=1,
            )
        )

    values = np.vstack(rows)
    statistics = ["center", "mean", "std", "p10", "p50", "p90"]
    columns = [
        f"{prefix}__b{band:02d}__{statistic}"
        for statistic in statistics
        for band in range(bands)
    ] + [f"{prefix}__valid_fraction"]

    return pd.DataFrame(values, columns=columns)

def attach_summary_to_rows(summary_df, patch_index, index_column, sensor_name):
    if index_column is None:
        if len(summary_df) != len(patch_index):
            raise RuntimeError(
                f"No {sensor_name} index column and row counts differ: "
                f"{len(summary_df)} vs {len(patch_index)}"
            )
        output = summary_df.copy()
        output.insert(0, "row_id", patch_index["row_id"].to_numpy())
        return output

    indices = pd.to_numeric(patch_index[index_column], errors="coerce")
    valid = indices.notna() & indices.ge(0) & indices.lt(len(summary_df))

    selected = pd.DataFrame(
        np.nan,
        index=np.arange(len(patch_index)),
        columns=summary_df.columns,
    )
    selected.loc[valid, :] = summary_df.iloc[
        indices.loc[valid].astype(int).to_numpy()
    ].to_numpy()

    return pd.concat(
        [
            patch_index[["row_id"]].reset_index(drop=True),
            selected.reset_index(drop=True),
        ],
        axis=1,
    )

patch_summary = None

if not RUN_PATCH_MODELS:
    print("Patch-summary models disabled.")

elif PATCH_SUMMARY_CACHE.exists():
    patch_summary = pd.read_csv(PATCH_SUMMARY_CACHE)
    print(
        "Loaded reusable patch-summary cache:",
        PATCH_SUMMARY_CACHE,
        "| shape=", patch_summary.shape,
    )

elif not (
    PATCH_INDEX_PATH.exists()
    and GEE_PATCH_NPZ.exists()
    and NICFI_PATCH_NPZ.exists()
):
    print("Patch inputs incomplete; patch models will be skipped.")
    print("Patch index:", PATCH_INDEX_PATH.exists())
    print("GEE NPZ:", GEE_PATCH_NPZ.exists())
    print("NICFI NPZ:", NICFI_PATCH_NPZ.exists())

else:
    patch_index = normalize_core_columns(
        read_csv_required(PATCH_INDEX_PATH, "patch index")
    )

    if "row_id" not in patch_index.columns:
        raise RuntimeError("Patch index must contain row_id.")

    # These are trusted project-generated NPZ files containing metadata objects.
    with np.load(
        GEE_PATCH_NPZ,
        mmap_mode="r",
        allow_pickle=True,
    ) as archive:
        gee_array = largest_numeric_4d_array(archive)
        gee_summary_all = summarize_patch_array(gee_array, "geePatch")

    with np.load(
        NICFI_PATCH_NPZ,
        mmap_mode="r",
        allow_pickle=True,
    ) as archive:
        nicfi_array = largest_numeric_4d_array(archive)
        nicfi_summary_all = summarize_patch_array(nicfi_array, "nicfiPatch")

    gee_index_column = find_index_column(patch_index, "gee")
    nicfi_index_column = find_index_column(patch_index, "nicfi")

    gee_by_row = attach_summary_to_rows(
        gee_summary_all, patch_index, gee_index_column, "GEE"
    )
    nicfi_by_row = attach_summary_to_rows(
        nicfi_summary_all, patch_index, nicfi_index_column, "NICFI"
    )

    patch_summary = gee_by_row.merge(
        nicfi_by_row,
        on="row_id",
        how="outer",
        validate="one_to_one",
    )
    patch_summary.to_csv(PATCH_SUMMARY_CACHE, index=False)

    with open(PATCH_CACHE_MANIFEST, "w") as file:
        json.dump(
            {
                "created": time.strftime("%Y-%m-%d %H:%M:%S"),
                "patch_index_path": str(PATCH_INDEX_PATH),
                "gee_patch_path": str(GEE_PATCH_NPZ),
                "nicfi_patch_path": str(NICFI_PATCH_NPZ),
                "chunk_size": PATCH_CHUNK_SIZE,
                "n_rows": len(patch_summary),
                "n_features": len(patch_summary.columns) - 1,
            },
            file,
            indent=2,
        )

    print("Saved reusable patch-summary cache:", PATCH_SUMMARY_CACHE)



BLOCK 06 — REUSABLE PATCH-SUMMARY CACHE
Loaded reusable patch-summary cache: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/05_model_runs/03_teacher_model_selection/derived_feature_cache/patch_summary_features_all_labelled_rows.csv | shape= (10312, 417)


## BLOCK 07 — Build candidate tables

In [ ]:
# ======================================================================================
# BLOCK 07 — BUILD AND AUDIT THE EIGHT CANDIDATE MODEL TABLES
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 07 — BUILD AND AUDIT THE EIGHT CANDIDATE MODEL TABLES")
print("=" * 100)

base_tables = {}

for model_name, path in MODEL_PATHS.items():
    table = normalize_core_columns(
        read_csv_required(path, model_name)
    )
    table["label_role"] = derive_label_role(table)
    table = table[
        table["land_cover_main5"].isin(TARGET_CLASSES)
    ].copy()
    table = table.merge(
        cluster_assignments,
        on="zone_id",
        how="left",
        validate="many_to_one",
    )
    table = table[table["cluster_id"].notna()].copy()
    table["cluster_id"] = table["cluster_id"].astype(int)
    table = table[
        ~table["zone_id"].isin(discarded_zones)
    ].copy()
    base_tables[model_name] = table

candidate_tables = dict(base_tables)

complexity_rank = {
    "M01_RF_GEE_POINT": 1,
    "M02_RF_GEE_S2_T24": 2,
    "M03_RF_GEE_S2_NICFI_T24": 3,
    "M04_RF_FULL_TABULAR_T12T24": 4,
    "PATCH_NICFI_STATS_ONLY": 5,
    "PATCH_GEE_STATS_ONLY": 6,
    "PATCH_GEE_NICFI_STATS": 7,
    "M02_PLUS_GEE_NICFI_PATCH_STATS": 8,
}

if patch_summary is not None:
    metadata_columns = [
        c for c in [
            "row_id", "unit_id", "zone_id", "target_yearmonth",
            "land_cover_main5", "label_role", "cluster_id",
        ]
        if c in base_tables["M01_RF_GEE_POINT"].columns
    ]

    metadata = (
        base_tables["M01_RF_GEE_POINT"][metadata_columns]
        .drop_duplicates("row_id")
    )

    patch_full = metadata.merge(
        patch_summary,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    gee_patch_columns = patch_columns(patch_full, "geePatch__")
    nicfi_patch_columns = patch_columns(patch_full, "nicfiPatch__")

    candidate_tables["PATCH_GEE_STATS_ONLY"] = patch_full[
        metadata_columns + gee_patch_columns
    ].copy()

    candidate_tables["PATCH_NICFI_STATS_ONLY"] = patch_full[
        metadata_columns + nicfi_patch_columns
    ].copy()

    candidate_tables["PATCH_GEE_NICFI_STATS"] = patch_full[
        metadata_columns + gee_patch_columns + nicfi_patch_columns
    ].copy()

    candidate_tables["M02_PLUS_GEE_NICFI_PATCH_STATS"] = (
        base_tables["M02_RF_GEE_S2_T24"]
        .merge(
            patch_summary,
            on="row_id",
            how="left",
            validate="one_to_one",
        )
    )
else:
    print("Patch summary unavailable: only M01–M04 will run.")

candidate_overview = []
candidate_feature_map = {}

for model_name, table in candidate_tables.items():
    features = model_feature_columns(model_name, table)
    candidate_feature_map[model_name] = features

    candidate_overview.append({
        "model": model_name,
        "n_rows": len(table),
        "n_features": len(features),
        "n_snapshot": int(table["label_role"].eq("snapshot").sum()),
        "n_trajectory": int(table["label_role"].eq("trajectory").sum()),
        "n_zones": int(table["zone_id"].nunique()),
        "n_clusters": int(table["cluster_id"].nunique()),
        "complexity_rank": complexity_rank[model_name],
        "first_feature": features[0],
        "last_feature": features[-1],
    })

candidate_overview_df = (
    pd.DataFrame(candidate_overview)
    .sort_values("complexity_rank")
)
display(candidate_overview_df)

candidate_overview_df.to_csv(
    TABLE_DIR / "candidate_model_overview.csv",
    index=False,
)

# Save the exact audited predictor list before model fitting.
with open(TABLE_DIR / "candidate_feature_columns.json", "w") as file:
    json.dump(candidate_feature_map, file, indent=2)

print("\nExcluded zones removed from every candidate table:", discarded_zones)



BLOCK 07 — BUILD AND AUDIT THE EIGHT CANDIDATE MODEL TABLES
Loaded M01_RF_GEE_POINT: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/03_model_tables/model_table_M01_GEE_POINT.csv | shape=(10312, 82)
Detected label column: label_main5 -> land_cover_main5


/tmp/ipykernel_3025/3165980065.py:49: DtypeWarning: Columns (396,397) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded M02_RF_GEE_S2_T24: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/03_model_tables/model_table_M02_GEE_S2_T24.csv | shape=(10312, 398)
Detected label column: label_main5 -> land_cover_main5


/tmp/ipykernel_3025/3165980065.py:49: DtypeWarning: Columns (396,397,1598,1599,1600,1601,1602,1603,1604,1605,1606,1607,1608,1609,1610,1611,1612,1613,1614,1615,1616,1617,1618,1619,1620,1621,1624,1625) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded M03_RF_GEE_S2_NICFI_T24: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/03_model_tables/model_table_M03_GEE_S2_NICFI_T24.csv | shape=(10312, 1630)
Detected label column: label_main5 -> land_cover_main5


/tmp/ipykernel_3025/3165980065.py:49: DtypeWarning: Columns (240,241,556,557,1158,1159,1160,1161,1162,1163,1164,1165,1166,1167,1168,1169,1172,1173,2378,2379,2380,2381,2382,2383,2384,2385,2386,2387,2388,2389,2390,2391,2392,2393,2394,2395,2396,2397,2398,2399,2400,2401,2404,2405) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Loaded M04_RF_FULL_TABULAR_T12T24: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/03_model_tables/model_table_M04_GEE_S2_NICFI_T12T24.csv | shape=(10312, 2410)
Detected label column: label_main5 -> land_cover_main5


,model,n_rows,n_features,n_snapshot,n_trajectory,n_zones,n_clusters,complexity_rank,first_feature,last_feature
0,M01_RF_GEE_POINT,9875,64,7013,2862,31,4,1,geept__A00,geept__A63
1,M02_RF_GEE_S2_T24,9875,382,7013,2862,31,4,2,geept__embedding_year,s2T24sum__fraction_valid_months
2,M03_RF_GEE_S2_NICFI_T24,9875,1588,7013,2862,31,4,3,geept__embedding_year,nicfiT24sum__n_no_image
3,M04_RF_FULL_TABULAR_T12T24,9875,2352,7013,2862,31,4,4,geept__embedding_year,nicfiT24sum__n_no_image
5,PATCH_NICFI_STATS_ONLY,9875,31,7013,2862,31,4,5,nicfiPatch__b00__center,nicfiPatch__valid_fraction
4,PATCH_GEE_STATS_ONLY,9875,385,7013,2862,31,4,6,geePatch__b00__center,geePatch__valid_fraction
6,PATCH_GEE_NICFI_STATS,9875,416,7013,2862,31,4,7,geePatch__b00__center,nicfiPatch__valid_fraction
7,M02_PLUS_GEE_NICFI_PATCH_STATS,9875,798,7013,2862,31,4,8,geept__embedding_year,nicfiPatch__valid_fraction



Excluded zones removed from every candidate table: ['G3001_Z146_dynamic_high_change', 'G3002_Z147_dynamic_high_change', 'G3005_Z150_dynamic_high_change']


## BLOCK 08a — Point-month validation on manual trajectories

Each teacher is trained only on retained snapshot observations. It is then evaluated on every manually labelled trajectory point-month.

This answers:

> Can a snapshot-trained teacher correctly classify the observed land-cover state at a dated trajectory observation?

Reported metrics include balanced accuracy, macro-F1, cropland precision, recall and F1, with unit-level bootstrap uncertainty.


In [ ]:
# ======================================================================================
# BLOCK 08 — TEACHER VALIDATION ON MONTHLY TRAJECTORY OBSERVATIONS
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 08 — TEACHER VALIDATION ON MONTHLY TRAJECTORY OBSERVATIONS")
print("=" * 100)

teacher_metrics = []
teacher_predictions = []
teacher_confusion = []
feature_manifests = []

for model_name, table in candidate_tables.items():
    feature_columns = candidate_feature_map[model_name]

    for feature in feature_columns:
        feature_manifests.append({
            "model": model_name,
            "feature": feature,
        })

    print("\n" + "=" * 100)
    print(model_name, "| features:", len(feature_columns))
    print("=" * 100)

    for cluster_id in sorted(table["cluster_id"].unique()):
        cluster_table = table[
            table["cluster_id"].eq(cluster_id)
        ].copy()

        train = cluster_table[
            cluster_table["label_role"].eq("snapshot")
        ].copy()
        test = cluster_table[
            cluster_table["label_role"].eq("trajectory")
        ].copy()

        # Strict location-level independence.
        test_units = set(test["unit_id"].astype(str))
        train = train[
            ~train["unit_id"].astype(str).isin(test_units)
        ].copy()

        status = "evaluated"
        reason = ""

        if len(train) < MIN_TEACHER_TRAIN_ROWS:
            status, reason = "skipped", "too_few_snapshot_training_rows"
        elif len(test) < MIN_TEACHER_TEST_ROWS:
            status, reason = "skipped", "too_few_trajectory_test_rows"
        elif train["land_cover_main5"].nunique() < MIN_TEACHER_TRAIN_CLASSES:
            status, reason = "skipped", "too_few_snapshot_training_classes"
        elif test["land_cover_main5"].nunique() < MIN_TEACHER_TEST_CLASSES:
            status, reason = "skipped", "too_few_trajectory_test_classes"

        common = {
            "model": model_name,
            "cluster_id": cluster_id,
            "status": status,
            "reason": reason,
            "n_features": len(feature_columns),
            "n_train": len(train),
            "n_test": len(test),
            "n_train_zones": int(train["zone_id"].nunique()),
            "n_test_zones": int(test["zone_id"].nunique()),
            "complexity_rank": complexity_rank[model_name],
        }

        if status == "skipped":
            teacher_metrics.append(common)
            print(f"Cluster {cluster_id}: SKIPPED — {reason}")
            continue

        model = make_rf()
        sample_weight = compute_sample_weight(
            "balanced",
            train["land_cover_main5"],
        )
        model.fit(
            train[feature_columns],
            train["land_cover_main5"],
            rf__sample_weight=sample_weight,
        )

        predictions = model.predict(test[feature_columns])
        probabilities = model.predict_proba(test[feature_columns])
        classes = list(model.named_steps["rf"].classes_)

        pred_df = test[
            [
                "row_id", "unit_id", "zone_id",
                "target_yearmonth", "cluster_id",
                "land_cover_main5",
            ]
        ].copy()
        pred_df["model"] = model_name
        pred_df["prediction"] = predictions
        pred_df["prediction_correct"] = (
            pred_df["prediction"].eq(pred_df["land_cover_main5"])
        )
        pred_df["max_probability"] = probabilities.max(axis=1)

        for index, class_name in enumerate(classes):
            pred_df[f"p_{class_name}"] = probabilities[:, index]

        teacher_predictions.append(pred_df)

        metrics = safe_multiclass_metrics(
            test["land_cover_main5"].to_numpy(),
            predictions,
        )
        metrics.update(grouped_bootstrap_metrics(pred_df))
        metrics.update(common)
        metrics.update({
            "status": "evaluated",
            "reason": "",
            "n_train_units": int(train["unit_id"].nunique()),
            "n_test_units": int(test["unit_id"].nunique()),
        })

        teacher_metrics.append(metrics)
        teacher_confusion.extend(
            confusion_long(
                test["land_cover_main5"].to_numpy(),
                predictions,
                {
                    "model": model_name,
                    "cluster_id": cluster_id,
                    "stage": "trajectory_validation",
                },
            )
        )

        print(
            f"Cluster {cluster_id}: "
            f"train={len(train)} | "
            f"test point-months={len(test)} | "
            f"BA={metrics['balanced_accuracy']:.3f} | "
            f"macroF1={metrics['macro_f1']:.3f} | "
            f"cropF1={metrics['cropland_f1']:.3f}"
        )

teacher_metrics_df = pd.DataFrame(teacher_metrics)
teacher_predictions_df = (
    pd.concat(teacher_predictions, ignore_index=True)
    if teacher_predictions else pd.DataFrame()
)
teacher_confusion_df = pd.DataFrame(teacher_confusion)
feature_manifest_df = pd.DataFrame(feature_manifests)

display(
    teacher_metrics_df[
        teacher_metrics_df["status"].eq("evaluated")
    ].sort_values(
        ["cluster_id", "balanced_accuracy"],
        ascending=[True, False],
    )
)



BLOCK 08 — TEACHER VALIDATION ON MONTHLY TRAJECTORY OBSERVATIONS

M01_RF_GEE_POINT | features: 64
Cluster 1: train=641 | test point-months=303 | BA=0.463 | macroF1=0.472 | cropF1=0.833
Cluster 2: train=1817 | test point-months=951 | BA=0.706 | macroF1=0.552 | cropF1=0.856
Cluster 3: train=781 | test point-months=353 | BA=0.571 | macroF1=0.450 | cropF1=0.757
Cluster 4: train=3658 | test point-months=1255 | BA=0.690 | macroF1=0.658 | cropF1=0.589

M02_RF_GEE_S2_T24 | features: 382
Cluster 1: train=641 | test point-months=303 | BA=0.461 | macroF1=0.469 | cropF1=0.826
Cluster 2: train=1817 | test point-months=951 | BA=0.641 | macroF1=0.545 | cropF1=0.832
Cluster 3: train=781 | test point-months=353 | BA=0.595 | macroF1=0.451 | cropF1=0.753
Cluster 4: train=3658 | test point-months=1255 | BA=0.673 | macroF1=0.630 | cropF1=0.616

M03_RF_GEE_S2_NICFI_T24 | features: 1588
Cluster 1: train=641 | test point-months=303 | BA=0.467 | macroF1=0.481 | cropF1=0.852
Cluster 2: train=1817 | test point-

,n,n_classes_true,accuracy,balanced_accuracy,macro_f1,cropland_n_true,cropland_n_pred,cropland_precision,cropland_recall,cropland_f1,...,status,reason,n_features,n_train,n_test,n_train_zones,n_test_zones,complexity_rank,n_train_units,n_test_units
28,303,5,0.825083,0.657287,0.668721,98,128,0.742188,0.969388,0.840708,...,evaluated,,798,641,303,4,4,8,641,63
24,303,5,0.798680,0.639927,0.649592,98,133,0.699248,0.948980,0.805195,...,evaluated,,416,641,303,4,4,7,641,63
16,303,5,0.755776,0.636622,0.643420,98,149,0.630872,0.959184,0.761134,...,evaluated,,385,641,303,4,4,6,641,63
8,303,5,0.831683,0.466896,0.481211,98,118,0.779661,0.938776,0.851852,...,evaluated,,1588,641,303,4,4,3,641,63
0,303,5,0.815182,0.463203,0.471963,98,130,0.730769,0.969388,0.833333,...,evaluated,,64,641,303,4,4,1,641,63
4,303,5,0.808581,0.460749,0.469084,98,132,0.719697,0.969388,0.826087,...,evaluated,,382,641,303,4,4,2,641,63
12,303,5,0.818482,0.451784,0.465142,98,122,0.770492,0.959184,0.854545,...,evaluated,,2352,641,303,4,4,4,641,63
20,303,5,0.729373,0.413772,0.409189,98,136,0.647059,0.897959,0.752137,...,evaluated,,31,641,303,4,4,5,641,63
17,951,4,0.801262,0.709454,0.565277,518,599,0.804674,0.930502,0.863026,...,evaluated,,385,1817,951,10,10,6,1817,213
1,951,4,0.789695,0.706401,0.551563,518,559,0.824687,0.889961,0.856082,...,evaluated,,64,1817,951,10,10,1,1817,213


## BLOCK 08b — Compact comparison of all eight candidate models

This block summarizes point-month performance for every model and cluster using the consistent codes **M01–M04** and **P01–P04**.

The full internal model names remain available in saved files, but short codes are used in displayed tables.

In [ ]:
# ======================================================================================
# BLOCK 08b — CLEAR MODEL PERFORMANCE SUMMARY BY CLUSTER
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 08b — MODEL PERFORMANCE SUMMARY BY CLUSTER")
print("=" * 100)

# Keep only successfully evaluated model-cluster combinations.
summary_metrics = teacher_metrics_df[
    teacher_metrics_df["status"].eq("evaluated")
].copy()

if summary_metrics.empty:
    raise RuntimeError(
        "No evaluated teacher-model results are available. "
        "Run Block 08 first."
    )


# ======================================================================================
# SHORT, AUTOMATIC MODEL NAMES
# ======================================================================================

MODEL_SHORT_NAMES = {
    "M01_RF_GEE_POINT": "M01 GEE point",
    "M02_RF_GEE_S2_T24": "M02 GEE+S2",
    "M03_RF_GEE_S2_NICFI_T24": "M03 GEE+S2+NICFI",
    "M04_RF_FULL_TABULAR_T12T24": "M04 full temporal",
    "PATCH_GEE_STATS_ONLY": "P01 GEE patch",
    "PATCH_NICFI_STATS_ONLY": "P02 NICFI patch",
    "PATCH_GEE_NICFI_STATS": "P03 both patches",
    "M02_PLUS_GEE_NICFI_PATCH_STATS": "P04 temporal+patch",
}

summary_metrics["model_short"] = (
    summary_metrics["model"]
    .map(MODEL_SHORT_NAMES)
    .fillna(summary_metrics["model"])
)


# ======================================================================================
# 1. MAIN COMPACT TABLE — ALL MODELS × CLUSTERS
# ======================================================================================

main_columns = [
    "cluster_id",
    "model_short",
    "n_features",
    "n_train",
    "n_test",
    "balanced_accuracy",
    "macro_f1",
    "cropland_precision",
    "cropland_recall",
    "cropland_f1",
]

compact_results = summary_metrics[main_columns].copy()

compact_results.columns = [
    "Cluster",
    "Model",
    "Features",
    "TrainN",
    "TestN",
    "BA",
    "MacroF1",
    "CropPrec",
    "CropRec",
    "CropF1",
]

for column in [
    "BA",
    "MacroF1",
    "CropPrec",
    "CropRec",
    "CropF1",
]:
    compact_results[column] = compact_results[column].round(3)

compact_results = compact_results.sort_values(
    ["Cluster", "BA", "CropF1"],
    ascending=[True, False, False],
)

print("\n" + "-" * 100)
print("ALL MODELS BY CLUSTER")
print("-" * 100)

display(compact_results)


# ======================================================================================
# 2. BALANCED ACCURACY MATRIX
# ======================================================================================

ba_matrix = (
    summary_metrics
    .pivot(
        index="model_short",
        columns="cluster_id",
        values="balanced_accuracy",
    )
    .rename_axis(index="Model", columns="Cluster")
    .round(3)
)

ba_matrix["Mean"] = ba_matrix.mean(axis=1).round(3)
ba_matrix["Min"] = ba_matrix.min(axis=1).round(3)
ba_matrix["Max"] = ba_matrix.max(axis=1).round(3)

ba_matrix = ba_matrix.sort_values(
    "Mean",
    ascending=False,
)

print("\n" + "-" * 100)
print("BALANCED ACCURACY BY CLUSTER")
print("-" * 100)

display(ba_matrix)


# ======================================================================================
# 3. CROPLAND F1 MATRIX
# ======================================================================================

crop_f1_matrix = (
    summary_metrics
    .pivot(
        index="model_short",
        columns="cluster_id",
        values="cropland_f1",
    )
    .rename_axis(index="Model", columns="Cluster")
    .round(3)
)

crop_f1_matrix["Mean"] = (
    crop_f1_matrix.mean(axis=1).round(3)
)

crop_f1_matrix["Min"] = (
    crop_f1_matrix.min(axis=1).round(3)
)

crop_f1_matrix["Max"] = (
    crop_f1_matrix.max(axis=1).round(3)
)

crop_f1_matrix = crop_f1_matrix.sort_values(
    "Mean",
    ascending=False,
)

print("\n" + "-" * 100)
print("CROPLAND F1 BY CLUSTER")
print("-" * 100)

display(crop_f1_matrix)


# ======================================================================================
# 4. MACRO-F1 MATRIX
# ======================================================================================

macro_f1_matrix = (
    summary_metrics
    .pivot(
        index="model_short",
        columns="cluster_id",
        values="macro_f1",
    )
    .rename_axis(index="Model", columns="Cluster")
    .round(3)
)

macro_f1_matrix["Mean"] = (
    macro_f1_matrix.mean(axis=1).round(3)
)

macro_f1_matrix["Min"] = (
    macro_f1_matrix.min(axis=1).round(3)
)

macro_f1_matrix["Max"] = (
    macro_f1_matrix.max(axis=1).round(3)
)

macro_f1_matrix = macro_f1_matrix.sort_values(
    "Mean",
    ascending=False,
)

print("\n" + "-" * 100)
print("MACRO-F1 BY CLUSTER")
print("-" * 100)

display(macro_f1_matrix)


# ======================================================================================
# 5. BEST MODEL PER CLUSTER FOR EACH METRIC
# ======================================================================================

best_rows = []

for cluster_id, cluster_results in summary_metrics.groupby(
    "cluster_id",
    sort=True,
):

    best_ba = cluster_results.loc[
        cluster_results["balanced_accuracy"].idxmax()
    ]

    best_macro = cluster_results.loc[
        cluster_results["macro_f1"].idxmax()
    ]

    best_crop = cluster_results.loc[
        cluster_results["cropland_f1"].idxmax()
    ]

    best_rows.append({
        "Cluster": int(cluster_id),

        "Best BA model": best_ba["model_short"],
        "Best BA": best_ba["balanced_accuracy"],

        "Best MacroF1 model": best_macro["model_short"],
        "Best MacroF1": best_macro["macro_f1"],

        "Best CropF1 model": best_crop["model_short"],
        "Best CropF1": best_crop["cropland_f1"],
    })

best_by_cluster = pd.DataFrame(best_rows)

for column in [
    "Best BA",
    "Best MacroF1",
    "Best CropF1",
]:
    best_by_cluster[column] = best_by_cluster[column].round(3)

print("\n" + "-" * 100)
print("BEST MODEL PER CLUSTER")
print("-" * 100)

display(best_by_cluster)


# ======================================================================================
# 6. OVERALL MODEL SUMMARY ACROSS CLUSTERS
# ======================================================================================
# Two averages are reported:
#
# - macro_cluster: every cluster receives equal weight;
# - weighted: clusters are weighted by their number of trajectory point-months.
#
# The equal-cluster mean is preferable when selecting a robust model across
# ecological clusters. The weighted mean is dominated by larger clusters.

overall_rows = []

for model_name, model_results in summary_metrics.groupby(
    "model_short",
    sort=False,
):

    weights = model_results["n_test"].to_numpy(dtype=float)

    overall_rows.append({
        "Model": model_name,
        "Clusters": model_results["cluster_id"].nunique(),
        "Features": int(model_results["n_features"].max()),

        "Mean BA": model_results["balanced_accuracy"].mean(),
        "Weighted BA": np.average(
            model_results["balanced_accuracy"],
            weights=weights,
        ),

        "Mean MacroF1": model_results["macro_f1"].mean(),
        "Weighted MacroF1": np.average(
            model_results["macro_f1"],
            weights=weights,
        ),

        "Mean CropF1": model_results["cropland_f1"].mean(),
        "Weighted CropF1": np.average(
            model_results["cropland_f1"],
            weights=weights,
        ),

        "Minimum BA": model_results["balanced_accuracy"].min(),
        "Minimum CropF1": model_results["cropland_f1"].min(),
    })

overall_summary = pd.DataFrame(overall_rows)

metric_columns = [
    "Mean BA",
    "Weighted BA",
    "Mean MacroF1",
    "Weighted MacroF1",
    "Mean CropF1",
    "Weighted CropF1",
    "Minimum BA",
    "Minimum CropF1",
]

overall_summary[metric_columns] = (
    overall_summary[metric_columns]
    .round(3)
)

overall_summary = overall_summary.sort_values(
    ["Mean BA", "Mean CropF1"],
    ascending=[False, False],
)

print("\n" + "-" * 100)
print("OVERALL MODEL SUMMARY ACROSS CLUSTERS")
print("-" * 100)

display(overall_summary)


# ======================================================================================
# 7. COMPACT UNCERTAINTY TABLE
# ======================================================================================

bootstrap_columns = [
    "cluster_id",
    "model_short",
    "balanced_accuracy",
    "balanced_accuracy_boot_lo",
    "balanced_accuracy_boot_hi",
    "cropland_f1",
    "cropland_f1_boot_lo",
    "cropland_f1_boot_hi",
]

available_bootstrap_columns = [
    column
    for column in bootstrap_columns
    if column in summary_metrics.columns
]

if len(available_bootstrap_columns) == len(bootstrap_columns):

    uncertainty_table = summary_metrics[
        bootstrap_columns
    ].copy()

    uncertainty_table["BA 95% CI"] = (
        uncertainty_table.apply(
            lambda row:
            f"{row['balanced_accuracy']:.3f} "
            f"[{row['balanced_accuracy_boot_lo']:.3f}, "
            f"{row['balanced_accuracy_boot_hi']:.3f}]",
            axis=1,
        )
    )

    uncertainty_table["CropF1 95% CI"] = (
        uncertainty_table.apply(
            lambda row:
            f"{row['cropland_f1']:.3f} "
            f"[{row['cropland_f1_boot_lo']:.3f}, "
            f"{row['cropland_f1_boot_hi']:.3f}]",
            axis=1,
        )
    )

    uncertainty_table = uncertainty_table[
        [
            "cluster_id",
            "model_short",
            "BA 95% CI",
            "CropF1 95% CI",
        ]
    ]

    uncertainty_table.columns = [
        "Cluster",
        "Model",
        "BA [95% CI]",
        "CropF1 [95% CI]",
    ]

    print("\n" + "-" * 100)
    print("BOOTSTRAP UNCERTAINTY")
    print("-" * 100)

    display(
        uncertainty_table.sort_values(
            ["Cluster", "Model"]
        )
    )

else:

    uncertainty_table = pd.DataFrame()

    print(
        "\nBootstrap confidence intervals are unavailable. "
        "This is expected when N_BOOTSTRAP=0."
    )


# ======================================================================================
# 8. SAVE ALL SUMMARY TABLES
# ======================================================================================

compact_results.to_csv(
    TABLE_DIR / "teacher_model_results_all_clusters.csv",
    index=False,
)

ba_matrix.to_csv(
    TABLE_DIR / "teacher_model_balanced_accuracy_matrix.csv",
)

crop_f1_matrix.to_csv(
    TABLE_DIR / "teacher_model_cropland_f1_matrix.csv",
)

macro_f1_matrix.to_csv(
    TABLE_DIR / "teacher_model_macro_f1_matrix.csv",
)

best_by_cluster.to_csv(
    TABLE_DIR / "teacher_model_best_per_cluster.csv",
    index=False,
)

overall_summary.to_csv(
    TABLE_DIR / "teacher_model_overall_summary.csv",
    index=False,
)

if not uncertainty_table.empty:

    uncertainty_table.to_csv(
        TABLE_DIR / "teacher_model_bootstrap_uncertainty.csv",
        index=False,
    )


print("\n" + "=" * 100)
print("BLOCK 08b SUMMARY")
print("=" * 100)

print(
    f"Evaluated models: "
    f"{summary_metrics['model'].nunique()}"
)

print(
    f"Evaluated clusters: "
    f"{summary_metrics['cluster_id'].nunique()}"
)

print(
    "\nBest mean balanced-accuracy model:"
)

best_overall_ba = overall_summary.iloc[0]

print(
    f"  {best_overall_ba['Model']} | "
    f"mean BA={best_overall_ba['Mean BA']:.3f}"
)

best_overall_crop = (
    overall_summary
    .sort_values(
        "Mean CropF1",
        ascending=False,
    )
    .iloc[0]
)

print(
    "\nBest mean cropland-F1 model:"
)

print(
    f"  {best_overall_crop['Model']} | "
    f"mean CropF1={best_overall_crop['Mean CropF1']:.3f}"
)


BLOCK 08b — MODEL PERFORMANCE SUMMARY BY CLUSTER

----------------------------------------------------------------------------------------------------
ALL MODELS BY CLUSTER
----------------------------------------------------------------------------------------------------


,Cluster,Model,Features,TrainN,TestN,BA,MacroF1,CropPrec,CropRec,CropF1
28,1,P04 temporal+patch,798,641,303,0.657,0.669,0.742,0.969,0.841
24,1,P03 both patches,416,641,303,0.640,0.650,0.699,0.949,0.805
16,1,P01 GEE patch,385,641,303,0.637,0.643,0.631,0.959,0.761
8,1,M03 GEE+S2+NICFI,1588,641,303,0.467,0.481,0.780,0.939,0.852
0,1,M01 GEE point,64,641,303,0.463,0.472,0.731,0.969,0.833
4,1,M02 GEE+S2,382,641,303,0.461,0.469,0.720,0.969,0.826
12,1,M04 full temporal,2352,641,303,0.452,0.465,0.770,0.959,0.855
20,1,P02 NICFI patch,31,641,303,0.414,0.409,0.647,0.898,0.752
17,2,P01 GEE patch,385,1817,951,0.709,0.565,0.805,0.931,0.863
1,2,M01 GEE point,64,1817,951,0.706,0.552,0.825,0.890,0.856



----------------------------------------------------------------------------------------------------
BALANCED ACCURACY BY CLUSTER
----------------------------------------------------------------------------------------------------


Cluster,1,2,3,4,Mean,Min,Max
Model,,,,,,,
P01 GEE patch,0.637,0.709,0.540,0.721,0.652,0.540,0.721
P03 both patches,0.640,0.675,0.574,0.707,0.649,0.574,0.707
P04 temporal+patch,0.657,0.653,0.585,0.686,0.645,0.585,0.686
M01 GEE point,0.463,0.706,0.571,0.690,0.607,0.463,0.706
M02 GEE+S2,0.461,0.641,0.595,0.673,0.592,0.461,0.673
M04 full temporal,0.452,0.656,0.529,0.712,0.587,0.452,0.712
M03 GEE+S2+NICFI,0.467,0.633,0.537,0.688,0.581,0.467,0.688
P02 NICFI patch,0.414,0.591,0.436,0.670,0.528,0.414,0.670



----------------------------------------------------------------------------------------------------
CROPLAND F1 BY CLUSTER
----------------------------------------------------------------------------------------------------


...


----------------------------------------------------------------------------------------------------
OVERALL MODEL SUMMARY ACROSS CLUSTERS
----------------------------------------------------------------------------------------------------


,Model,Clusters,Features,Mean BA,Weighted BA,Mean MacroF1,Weighted MacroF1,Mean CropF1,Weighted CropF1,Minimum BA,Minimum CropF1
4,P01 GEE patch,4,385,0.652,0.686,0.579,0.609,0.736,0.713,0.540,0.579
6,P03 both patches,4,416,0.649,0.673,0.576,0.599,0.752,0.723,0.574,0.599
7,P04 temporal+patch,4,798,0.645,0.660,0.572,0.583,0.766,0.737,0.585,0.619
0,M01 GEE point,4,64,0.608,0.657,0.533,0.578,0.759,0.724,0.463,0.589
1,M02 GEE+S2,4,382,0.593,0.630,0.524,0.563,0.757,0.727,0.461,0.616
3,M04 full temporal,4,2352,0.587,0.643,0.514,0.569,0.720,0.694,0.452,0.588
2,M03 GEE+S2+NICFI,4,1588,0.581,0.628,0.512,0.555,0.729,0.700,0.467,0.605
5,P02 NICFI patch,4,31,0.528,0.588,0.419,0.460,0.617,0.594,0.414,0.485



----------------------------------------------------------------------------------------------------
BOOTSTRAP UNCERTAINTY
----------------------------------------------------------------------------------------------------


,Cluster,Model,BA [95% CI],CropF1 [95% CI]
0,1,M01 GEE point,"0.463 [0.403, 0.645]","0.833 [0.705, 0.895]"
4,1,M02 GEE+S2,"0.461 [0.397, 0.634]","0.826 [0.697, 0.889]"
8,1,M03 GEE+S2+NICFI,"0.467 [0.402, 0.639]","0.852 [0.719, 0.930]"
12,1,M04 full temporal,"0.452 [0.390, 0.618]","0.855 [0.748, 0.912]"
16,1,P01 GEE patch,"0.637 [0.558, 0.839]","0.761 [0.607, 0.849]"
20,1,P02 NICFI patch,"0.414 [0.357, 0.568]","0.752 [0.568, 0.839]"
24,1,P03 both patches,"0.640 [0.572, 0.850]","0.805 [0.674, 0.871]"
28,1,P04 temporal+patch,"0.657 [0.590, 0.871]","0.841 [0.727, 0.901]"
1,2,M01 GEE point,"0.706 [0.558, 0.788]","0.856 [0.805, 0.899]"
5,2,M02 GEE+S2,"0.641 [0.558, 0.692]","0.832 [0.776, 0.874]"



BLOCK 08b SUMMARY
Evaluated models: 8
Evaluated clusters: 4

Best mean balanced-accuracy model:
  P01 GEE patch | mean BA=0.652

Best mean cropland-F1 model:
  P04 temporal+patch | mean CropF1=0.766


## BLOCK 08c — Full trajectory-behaviour diagnostics

Predictions are grouped by `unit_id` and ordered through time. The block compares:

- sequence agreement;
- agreement specifically on changing units;
- detection of any trajectory change;
- cropland presence over the trajectory;
- excess predicted transitions.

This block is diagnostic and does not itself choose the operational teacher.

In [ ]:
# ======================================================================================
# BLOCK 08c — COMPARE POINT-MONTH AND TRAJECTORY PERFORMANCE
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 08c — COMPARE POINT-MONTH AND TRAJECTORY PERFORMANCE")
print("=" * 100)

if teacher_predictions_df.empty:
    raise RuntimeError(
        "teacher_predictions_df is empty. Run Block 08 first."
    )

MODEL_CODES = {
    "M01_RF_GEE_POINT": "M01",
    "M02_RF_GEE_S2_T24": "M02",
    "M03_RF_GEE_S2_NICFI_T24": "M03",
    "M04_RF_FULL_TABULAR_T12T24": "M04",
    "PATCH_GEE_STATS_ONLY": "P01",
    "PATCH_NICFI_STATS_ONLY": "P02",
    "PATCH_GEE_NICFI_STATS": "P03",
    "M02_PLUS_GEE_NICFI_PATCH_STATS": "P04",
}


# ======================================================================================
# DATE HELPERS
# ======================================================================================

def parse_yearmonth(values):

    text = (
        values
        .astype(str)
        .str.strip()
        .str.replace("_", "-", regex=False)
    )

    parsed = pd.to_datetime(
        text,
        errors="coerce",
    )

    missing = parsed.isna()

    if missing.any():

        compact = (
            text.loc[missing]
            .str.replace("-", "", regex=False)
            .str.slice(0, 6)
        )

        parsed.loc[missing] = pd.to_datetime(
            compact,
            format="%Y%m",
            errors="coerce",
        )

    return parsed


def month_difference(date_a, date_b):

    if pd.isna(date_a) or pd.isna(date_b):
        return np.nan

    return (
        (date_a.year - date_b.year) * 12
        + date_a.month
        - date_b.month
    )


def safe_binary_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=bool)
    y_pred = np.asarray(y_pred, dtype=bool)

    return {
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
    }


# ======================================================================================
# PREPARE TRAJECTORY PREDICTIONS
# ======================================================================================

trajectory_predictions = teacher_predictions_df.copy()

trajectory_predictions["date"] = parse_yearmonth(
    trajectory_predictions["target_yearmonth"]
)

trajectory_predictions = trajectory_predictions[
    trajectory_predictions["date"].notna()
].copy()

trajectory_predictions["model_code"] = (
    trajectory_predictions["model"]
    .map(MODEL_CODES)
    .fillna(trajectory_predictions["model"])
)

trajectory_predictions = (
    trajectory_predictions
    .sort_values(
        [
            "model",
            "cluster_id",
            "unit_id",
            "date",
        ]
    )
    .drop_duplicates(
        [
            "model",
            "cluster_id",
            "unit_id",
            "date",
        ],
        keep="first",
    )
)


# ======================================================================================
# PER-UNIT TRAJECTORY METRICS
# ======================================================================================

unit_rows = []

for (
    model_name,
    model_code,
    cluster_id,
    unit_id,
), unit_df in trajectory_predictions.groupby(
    [
        "model",
        "model_code",
        "cluster_id",
        "unit_id",
    ],
    sort=False,
):

    unit_df = (
        unit_df
        .sort_values("date")
        .copy()
    )

    if unit_df["date"].nunique() < 2:
        continue

    observed = (
        unit_df["land_cover_main5"]
        .astype(str)
        .to_numpy()
    )

    predicted = (
        unit_df["prediction"]
        .astype(str)
        .to_numpy()
    )

    sequence_agreement = float(
        np.mean(observed == predicted)
    )

    observed_transitions = int(
        np.sum(observed[1:] != observed[:-1])
    )

    predicted_transitions = int(
        np.sum(predicted[1:] != predicted[:-1])
    )

    observed_any_change = observed_transitions > 0
    predicted_any_change = predicted_transitions > 0

    observed_crop_ever = bool(
        np.any(observed == CROP_CLASS)
    )

    predicted_crop_ever = bool(
        np.any(predicted == CROP_CLASS)
    )

    observed_crop_dates = (
        unit_df.loc[
            unit_df["land_cover_main5"].eq(CROP_CLASS),
            "date",
        ]
        .sort_values()
    )

    predicted_crop_dates = (
        unit_df.loc[
            unit_df["prediction"].eq(CROP_CLASS),
            "date",
        ]
        .sort_values()
    )

    first_observed_crop = (
        observed_crop_dates.iloc[0]
        if len(observed_crop_dates)
        else pd.NaT
    )

    first_predicted_crop = (
        predicted_crop_dates.iloc[0]
        if len(predicted_crop_dates)
        else pd.NaT
    )

    onset_error_months = month_difference(
        first_predicted_crop,
        first_observed_crop,
    )

    unit_rows.append({
        "model": model_name,
        "model_code": model_code,
        "cluster_id": int(cluster_id),
        "unit_id": unit_id,

        "n_dates": int(
            unit_df["date"].nunique()
        ),

        "sequence_agreement": sequence_agreement,

        "observed_any_change": observed_any_change,
        "predicted_any_change": predicted_any_change,

        "observed_crop_ever": observed_crop_ever,
        "predicted_crop_ever": predicted_crop_ever,

        "first_observed_crop": first_observed_crop,
        "first_predicted_crop": first_predicted_crop,

        "crop_onset_error_months": onset_error_months,

        "crop_onset_absolute_error_months": (
            abs(onset_error_months)
            if pd.notna(onset_error_months)
            else np.nan
        ),

        "observed_transitions": observed_transitions,
        "predicted_transitions": predicted_transitions,

        "excess_transitions": (
            predicted_transitions
            - observed_transitions
        ),

        "has_excess_transitions": (
            predicted_transitions
            > observed_transitions
        ),
    })


trajectory_unit_metrics_df = pd.DataFrame(
    unit_rows
)

if trajectory_unit_metrics_df.empty:
    raise RuntimeError(
        "No trajectories with at least two dates were available."
    )


# ======================================================================================
# MODEL × CLUSTER TRAJECTORY SUMMARY
# ======================================================================================

trajectory_summary_rows = []

for (
    model_name,
    model_code,
    cluster_id,
), group in trajectory_unit_metrics_df.groupby(
    [
        "model",
        "model_code",
        "cluster_id",
    ],
    sort=True,
):

    crop_presence_metrics = safe_binary_metrics(
        group["observed_crop_ever"],
        group["predicted_crop_ever"],
    )

    change_metrics = safe_binary_metrics(
        group["observed_any_change"],
        group["predicted_any_change"],
    )

    changing_units = group[
        group["observed_any_change"]
    ].copy()

    stable_units = group[
        ~group["observed_any_change"]
    ].copy()

    onset_comparable = group[
        group["crop_onset_absolute_error_months"]
        .notna()
    ].copy()

    trajectory_summary_rows.append({
        "model": model_name,
        "model_code": model_code,
        "cluster_id": int(cluster_id),

        "n_units": int(
            group["unit_id"].nunique()
        ),

        "n_changing_units": int(
            changing_units["unit_id"].nunique()
        ),

        "n_stable_units": int(
            stable_units["unit_id"].nunique()
        ),

        # Overall sequence agreement
        "mean_sequence_agreement": float(
            group["sequence_agreement"].mean()
        ),

        "median_sequence_agreement": float(
            group["sequence_agreement"].median()
        ),

        # Split stable versus changing trajectories
        "mean_sequence_agreement_changing": (
            float(
                changing_units[
                    "sequence_agreement"
                ].mean()
            )
            if len(changing_units)
            else np.nan
        ),

        "mean_sequence_agreement_stable": (
            float(
                stable_units[
                    "sequence_agreement"
                ].mean()
            )
            if len(stable_units)
            else np.nan
        ),

        # Ever-cropland detection
        "crop_presence_precision":
            crop_presence_metrics["precision"],

        "crop_presence_recall":
            crop_presence_metrics["recall"],

        "crop_presence_f1":
            crop_presence_metrics["f1"],

        # Any-change detection
        "change_precision":
            change_metrics["precision"],

        "change_recall":
            change_metrics["recall"],

        "change_f1":
            change_metrics["f1"],

        # Cropland-onset timing
        "n_onset_comparable": int(
            len(onset_comparable)
        ),

        "median_crop_onset_absolute_error_months": (
            float(
                onset_comparable[
                    "crop_onset_absolute_error_months"
                ].median()
            )
            if len(onset_comparable)
            else np.nan
        ),

        "mean_crop_onset_absolute_error_months": (
            float(
                onset_comparable[
                    "crop_onset_absolute_error_months"
                ].mean()
            )
            if len(onset_comparable)
            else np.nan
        ),

        "crop_onset_within_6_months": (
            float(
                onset_comparable[
                    "crop_onset_absolute_error_months"
                ]
                .le(6)
                .mean()
            )
            if len(onset_comparable)
            else np.nan
        ),

        "crop_onset_within_12_months": (
            float(
                onset_comparable[
                    "crop_onset_absolute_error_months"
                ]
                .le(12)
                .mean()
            )
            if len(onset_comparable)
            else np.nan
        ),

        # Temporal instability
        "mean_observed_transitions": float(
            group[
                "observed_transitions"
            ].mean()
        ),

        "mean_predicted_transitions": float(
            group[
                "predicted_transitions"
            ].mean()
        ),

        "mean_excess_transitions": float(
            group[
                "excess_transitions"
            ].mean()
        ),

        "fraction_units_with_excess_transitions": float(
            group[
                "has_excess_transitions"
            ].mean()
        ),
    })


trajectory_summary_df = pd.DataFrame(
    trajectory_summary_rows
)


# ======================================================================================
# ADD POINT-MONTH METRICS FROM BLOCK 08
# ======================================================================================

pointmonth_summary = (
    teacher_metrics_df[
        teacher_metrics_df["status"].eq("evaluated")
    ][
        [
            "model",
            "cluster_id",
            "balanced_accuracy",
            "macro_f1",
            "cropland_precision",
            "cropland_recall",
            "cropland_f1",
        ]
    ]
    .copy()
)

comparison_df = trajectory_summary_df.merge(
    pointmonth_summary,
    on=[
        "model",
        "cluster_id",
    ],
    how="left",
    validate="one_to_one",
)


# ======================================================================================
# COMPACT COMPARISON TABLE
# ======================================================================================

comparison_table = comparison_df[
    [
        "cluster_id",
        "model_code",

        "n_units",
        "n_changing_units",

        "balanced_accuracy",
        "cropland_f1",

        "mean_sequence_agreement",
        "mean_sequence_agreement_changing",

        "crop_presence_f1",
        "change_f1",

        "median_crop_onset_absolute_error_months",
        "crop_onset_within_12_months",

        "fraction_units_with_excess_transitions",
    ]
].copy()

comparison_table.columns = [
    "Cl",
    "Model",

    "Units",
    "ChangingN",

    "PointBA",
    "PointCropF1",

    "SeqAgree",
    "SeqAgreeChanging",

    "CropEverF1",
    "ChangeF1",

    "OnsetMAE",
    "Onset±12m",

    "ExcessTransitions",
]

for column in [
    "PointBA",
    "PointCropF1",
    "SeqAgree",
    "SeqAgreeChanging",
    "CropEverF1",
    "ChangeF1",
    "Onset±12m",
    "ExcessTransitions",
]:

    comparison_table[column] = (
        comparison_table[column]
        .round(3)
    )

comparison_table["OnsetMAE"] = (
    comparison_table["OnsetMAE"]
    .round(1)
)

comparison_table = comparison_table.sort_values(
    [
        "Cl",
        "SeqAgreeChanging",
        "ChangeF1",
        "OnsetMAE",
    ],
    ascending=[
        True,
        False,
        False,
        True,
    ],
    na_position="last",
)


print("\n" + "-" * 110)
print("POINT-MONTH VERSUS TRAJECTORY PERFORMANCE")
print("-" * 110)

display(
    comparison_table
)


# ======================================================================================
# RANK MATRICES FOR QUICK VISUAL COMPARISON
# ======================================================================================

def metric_matrix(
    source_df,
    value_column,
    title,
    higher_is_better=True,
):

    matrix = (
        source_df
        .pivot(
            index="model_code",
            columns="cluster_id",
            values=value_column,
        )
        .rename_axis(
            index="Model",
            columns="Cluster",
        )
        .round(3)
    )

    matrix["Mean"] = (
        matrix.mean(axis=1).round(3)
    )

    matrix = matrix.sort_values(
        "Mean",
        ascending=not higher_is_better,
    )

    print("\n" + "-" * 100)
    print(title)
    print("-" * 100)

    display(matrix)

    return matrix


point_ba_matrix = metric_matrix(
    comparison_df,
    "balanced_accuracy",
    "POINT-MONTH BALANCED ACCURACY",
    higher_is_better=True,
)

changing_sequence_matrix = metric_matrix(
    comparison_df,
    "mean_sequence_agreement_changing",
    "SEQUENCE AGREEMENT — CHANGING UNITS",
    higher_is_better=True,
)

change_f1_matrix = metric_matrix(
    comparison_df,
    "change_f1",
    "TRAJECTORY CHANGE-DETECTION F1",
    higher_is_better=True,
)

onset_error_matrix = metric_matrix(
    comparison_df,
    "median_crop_onset_absolute_error_months",
    "MEDIAN CROPLAND-ONSET ERROR IN MONTHS",
    higher_is_better=False,
)

instability_matrix = metric_matrix(
    comparison_df,
    "fraction_units_with_excess_transitions",
    "FRACTION OF UNITS WITH EXCESS PREDICTED TRANSITIONS",
    higher_is_better=False,
)


# ======================================================================================
# SAVE OUTPUTS
# ======================================================================================

trajectory_unit_metrics_df.to_csv(
    TABLE_DIR / "trajectory_diagnostics_by_unit.csv",
    index=False,
)

trajectory_summary_df.to_csv(
    TABLE_DIR / "trajectory_diagnostics_by_model_cluster.csv",
    index=False,
)

comparison_df.to_csv(
    TABLE_DIR / "pointmonth_trajectory_model_comparison_full.csv",
    index=False,
)

comparison_table.to_csv(
    TABLE_DIR / "pointmonth_trajectory_model_comparison_compact.csv",
    index=False,
)

point_ba_matrix.to_csv(
    TABLE_DIR / "matrix_pointmonth_balanced_accuracy.csv",
)

changing_sequence_matrix.to_csv(
    TABLE_DIR / "matrix_sequence_agreement_changing_units.csv",
)

change_f1_matrix.to_csv(
    TABLE_DIR / "matrix_change_detection_f1.csv",
)

onset_error_matrix.to_csv(
    TABLE_DIR / "matrix_crop_onset_error_months.csv",
)

instability_matrix.to_csv(
    TABLE_DIR / "matrix_excess_transition_fraction.csv",
)


print("\n" + "=" * 100)
print("BLOCK 08c SUMMARY")
print("=" * 100)

print(
    "This block is diagnostic only."
)

print(
    "It does not yet change the selected teachers."
)

print(
    "Use it first to see whether point-month winners are also "
    "trajectory winners."
)


BLOCK 08c — COMPARE POINT-MONTH AND TRAJECTORY PERFORMANCE

--------------------------------------------------------------------------------------------------------------
POINT-MONTH VERSUS TRAJECTORY PERFORMANCE
--------------------------------------------------------------------------------------------------------------


,Cl,Model,Units,ChangingN,PointBA,PointCropF1,SeqAgree,SeqAgreeChanging,CropEverF1,ChangeF1,OnsetMAE,Onset±12m,ExcessTransitions
12,1,M03,46,24,0.467,0.852,0.846,0.838,0.829,0.809,0.0,0.941,0.130
0,1,M01,46,24,0.463,0.833,0.836,0.832,0.837,0.756,0.0,0.833,0.152
8,1,M02,46,24,0.461,0.826,0.836,0.813,0.810,0.698,0.0,0.882,0.152
16,1,M04,46,24,0.452,0.855,0.819,0.801,0.829,0.667,0.0,0.941,0.283
4,1,P04,46,24,0.657,0.841,0.837,0.794,0.842,0.791,0.0,0.750,0.130
24,1,P01,46,24,0.637,0.761,0.785,0.766,0.800,0.696,0.0,0.722,0.152
28,1,P02,46,24,0.414,0.752,0.729,0.751,0.714,0.706,0.0,0.667,0.283
20,1,P03,46,24,0.640,0.805,0.805,0.748,0.789,0.732,0.0,0.600,0.152
17,2,M04,174,82,0.656,0.807,0.760,0.764,0.874,0.711,0.0,0.741,0.328
13,2,M03,174,82,0.633,0.786,0.753,0.743,0.878,0.697,0.0,0.713,0.316



----------------------------------------------------------------------------------------------------
POINT-MONTH BALANCED ACCURACY
----------------------------------------------------------------------------------------------------


Cluster,1,2,3,4,Mean
Model,,,,,
P01,0.637,0.709,0.540,0.721,0.652
P03,0.640,0.675,0.574,0.707,0.649
P04,0.657,0.653,0.585,0.686,0.645
M01,0.463,0.706,0.571,0.690,0.607
M02,0.461,0.641,0.595,0.673,0.592
M04,0.452,0.656,0.529,0.712,0.587
M03,0.467,0.633,0.537,0.688,0.581
P02,0.414,0.591,0.436,0.670,0.528



----------------------------------------------------------------------------------------------------
SEQUENCE AGREEMENT — CHANGING UNITS
----------------------------------------------------------------------------------------------------


...


----------------------------------------------------------------------------------------------------
MEDIAN CROPLAND-ONSET ERROR IN MONTHS
----------------------------------------------------------------------------------------------------


Cluster,1,2,3,4,Mean
Model,,,,,
M01,0.0,0.0,0.0,0.0,0.000
M02,0.0,0.0,0.0,0.0,0.000
M03,0.0,0.0,0.0,0.0,0.000
P01,0.0,0.0,0.0,0.0,0.000
P04,0.0,0.0,0.0,0.0,0.000
P03,0.0,0.0,0.0,0.0,0.000
P02,0.0,0.0,0.0,8.5,2.125
M04,0.0,0.0,0.0,11.0,2.750



----------------------------------------------------------------------------------------------------
FRACTION OF UNITS WITH EXCESS PREDICTED TRANSITIONS
----------------------------------------------------------------------------------------------------


Cluster,1,2,3,4,Mean
Model,,,,,
P01,0.152,0.149,0.217,0.263,0.195
P03,0.152,0.184,0.217,0.298,0.213
M01,0.152,0.207,0.229,0.273,0.215
P04,0.130,0.224,0.241,0.332,0.232
M02,0.152,0.293,0.325,0.371,0.285
M03,0.130,0.316,0.458,0.410,0.328
M04,0.283,0.328,0.458,0.454,0.381
P02,0.283,0.471,0.747,0.566,0.517



BLOCK 08c SUMMARY
This block is diagnostic only.
It does not yet change the selected teachers.
Use it first to see whether point-month winners are also trajectory winners.


## BLOCK 08d — Annual cropland-transition validation

Monthly predictions are aggregated to one representative observed and predicted state per unit-year. Consecutive annual pairs are classified as:

- stable non-cropland;
- cropland gain;
- stable cropland;
- cropland loss.

This is the most directly relevant diagnostic for annual trajectory pseudo-labels. Gain/loss support must always be considered when interpreting F1 values.

In [ ]:
# ======================================================================================
# BLOCK 08d — ANNUAL CROPLAND-TRANSITION VALIDATION
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 08d — ANNUAL CROPLAND-TRANSITION VALIDATION")
print("=" * 100)

# This block reuses the trajectory point-month predictions from Block 08.
# No model is retrained.

if teacher_predictions_df.empty:
    raise RuntimeError(
        "teacher_predictions_df is empty. Run Block 08 first."
    )


# ======================================================================================
# SETTINGS
# ======================================================================================

MODEL_CODES = {
    "M01_RF_GEE_POINT": "M01",
    "M02_RF_GEE_S2_T24": "M02",
    "M03_RF_GEE_S2_NICFI_T24": "M03",
    "M04_RF_FULL_TABULAR_T12T24": "M04",
    "PATCH_GEE_STATS_ONLY": "P01",
    "PATCH_NICFI_STATS_ONLY": "P02",
    "PATCH_GEE_NICFI_STATS": "P03",
    "M02_PLUS_GEE_NICFI_PATCH_STATS": "P04",
}

ANNUAL_TRANSITION_CLASSES = [
    "stable_non_crop",
    "gain_crop",
    "stable_crop",
    "loss_crop",
]

# Only compare consecutive calendar years.
# Set False to compare any successive observed years, including gaps.
REQUIRE_CONSECUTIVE_YEARS = True

# Minimum support for interpreting a gain/loss F1.
MIN_TRANSITION_SUPPORT = 5


# ======================================================================================
# DATE PARSING
# ======================================================================================

def parse_target_yearmonth(values):
    """
    Convert common target_yearmonth formats to pandas timestamps.

    Handles:
      2023-05
      2023_05
      202305
      2023-05-01
    """

    text = (
        values
        .astype(str)
        .str.strip()
        .str.replace("_", "-", regex=False)
    )

    parsed = pd.to_datetime(
        text,
        errors="coerce",
    )

    missing = parsed.isna()

    if missing.any():

        compact = (
            text.loc[missing]
            .str.replace("-", "", regex=False)
            .str.slice(0, 6)
        )

        parsed.loc[missing] = pd.to_datetime(
            compact,
            format="%Y%m",
            errors="coerce",
        )

    return parsed


# ======================================================================================
# ANNUAL STATE SELECTION
# ======================================================================================

def representative_annual_state(
    group,
    label_column,
):
    """
    Select one representative state for a unit-year.

    Rule:
      1. use the most frequent class within the year;
      2. if several classes tie, use the class at the latest observed month.

    The same observation dates are used for truth and prediction.
    """

    values = (
        group[label_column]
        .dropna()
        .astype(str)
    )

    if values.empty:
        return np.nan

    counts = values.value_counts()
    maximum_count = counts.max()

    tied_classes = set(
        counts[
            counts.eq(maximum_count)
        ].index
    )

    latest_tied_row = (
        group[
            group[label_column]
            .astype(str)
            .isin(tied_classes)
        ]
        .sort_values("date")
        .iloc[-1]
    )

    return str(
        latest_tied_row[label_column]
    )


# ======================================================================================
# TRANSITION HELPERS
# ======================================================================================

def cropland_transition_type(
    state_from,
    state_to,
):
    """
    Convert two annual land-cover states to one cropland transition class.
    """

    from_crop = state_from == CROP_CLASS
    to_crop = state_to == CROP_CLASS

    if not from_crop and not to_crop:
        return "stable_non_crop"

    if not from_crop and to_crop:
        return "gain_crop"

    if from_crop and to_crop:
        return "stable_crop"

    return "loss_crop"


def safe_binary_transition_metrics(
    truth,
    prediction,
    positive_class,
):
    """
    Precision, recall and F1 for one transition type versus all others.
    """

    truth_binary = (
        np.asarray(truth) == positive_class
    )

    prediction_binary = (
        np.asarray(prediction) == positive_class
    )

    return {
        "support": int(
            truth_binary.sum()
        ),
        "predicted_n": int(
            prediction_binary.sum()
        ),
        "precision": precision_score(
            truth_binary,
            prediction_binary,
            zero_division=0,
        ),
        "recall": recall_score(
            truth_binary,
            prediction_binary,
            zero_division=0,
        ),
        "f1": f1_score(
            truth_binary,
            prediction_binary,
            zero_division=0,
        ),
    }


# ======================================================================================
# PREPARE POINT-MONTH PREDICTIONS
# ======================================================================================

annual_input = teacher_predictions_df.copy()

required_columns = {
    "model",
    "cluster_id",
    "unit_id",
    "target_yearmonth",
    "land_cover_main5",
    "prediction",
}

missing_columns = sorted(
    required_columns
    - set(annual_input.columns)
)

if missing_columns:
    raise RuntimeError(
        "Missing required columns in teacher_predictions_df: "
        f"{missing_columns}"
    )

annual_input["date"] = parse_target_yearmonth(
    annual_input["target_yearmonth"]
)

n_invalid_dates = int(
    annual_input["date"].isna().sum()
)

if n_invalid_dates:
    print(
        f"Removing {n_invalid_dates} rows with invalid target_yearmonth."
    )

annual_input = annual_input[
    annual_input["date"].notna()
].copy()

annual_input["year"] = (
    annual_input["date"]
    .dt.year
    .astype(int)
)

annual_input["model_code"] = (
    annual_input["model"]
    .map(MODEL_CODES)
    .fillna(annual_input["model"])
)

# Ensure one record per model-cluster-unit-date.
annual_input = (
    annual_input
    .sort_values(
        [
            "model",
            "cluster_id",
            "unit_id",
            "date",
        ]
    )
    .drop_duplicates(
        [
            "model",
            "cluster_id",
            "unit_id",
            "date",
        ],
        keep="first",
    )
)


# ======================================================================================
# DERIVE ONE OBSERVED AND PREDICTED STATE PER UNIT-YEAR
# ======================================================================================

annual_state_rows = []

annual_group_columns = [
    "model",
    "model_code",
    "cluster_id",
    "unit_id",
    "year",
]

for group_values, year_df in annual_input.groupby(
    annual_group_columns,
    sort=False,
):

    (
        model_name,
        model_code,
        cluster_id,
        unit_id,
        year,
    ) = group_values

    year_df = (
        year_df
        .sort_values("date")
        .copy()
    )

    observed_annual_state = (
        representative_annual_state(
            year_df,
            "land_cover_main5",
        )
    )

    predicted_annual_state = (
        representative_annual_state(
            year_df,
            "prediction",
        )
    )

    annual_state_rows.append({
        "model": model_name,
        "model_code": model_code,
        "cluster_id": int(cluster_id),
        "unit_id": unit_id,
        "year": int(year),

        "n_observed_dates": int(
            year_df["date"].nunique()
        ),

        "first_month": int(
            year_df["date"].dt.month.min()
        ),

        "last_month": int(
            year_df["date"].dt.month.max()
        ),

        "observed_annual_state":
            observed_annual_state,

        "predicted_annual_state":
            predicted_annual_state,

        "observed_annual_crop": bool(
            observed_annual_state == CROP_CLASS
        ),

        "predicted_annual_crop": bool(
            predicted_annual_state == CROP_CLASS
        ),
    })


annual_state_df = pd.DataFrame(
    annual_state_rows
)

if annual_state_df.empty:
    raise RuntimeError(
        "No annual states could be derived."
    )


# ======================================================================================
# ANNUAL STATE METRICS BEFORE TRANSITIONS
# ======================================================================================

annual_state_summary_rows = []

for (
    model_name,
    model_code,
    cluster_id,
), group in annual_state_df.groupby(
    [
        "model",
        "model_code",
        "cluster_id",
    ],
    sort=True,
):

    annual_state_summary_rows.append({
        "model": model_name,
        "model_code": model_code,
        "cluster_id": int(cluster_id),

        "n_unit_years": int(
            len(group)
        ),

        "n_units": int(
            group["unit_id"].nunique()
        ),

        "annual_state_accuracy": accuracy_score(
            group["observed_annual_state"],
            group["predicted_annual_state"],
        ),

        "annual_state_balanced_accuracy":
            safe_multiclass_metrics(
                group["observed_annual_state"],
                group["predicted_annual_state"],
            )["balanced_accuracy"],

        "annual_crop_precision": precision_score(
            group["observed_annual_crop"],
            group["predicted_annual_crop"],
            zero_division=0,
        ),

        "annual_crop_recall": recall_score(
            group["observed_annual_crop"],
            group["predicted_annual_crop"],
            zero_division=0,
        ),

        "annual_crop_f1": f1_score(
            group["observed_annual_crop"],
            group["predicted_annual_crop"],
            zero_division=0,
        ),
    })


annual_state_summary_df = pd.DataFrame(
    annual_state_summary_rows
)


# ======================================================================================
# BUILD ANNUAL TRANSITIONS
# ======================================================================================

annual_transition_rows = []

trajectory_group_columns = [
    "model",
    "model_code",
    "cluster_id",
    "unit_id",
]

for group_values, unit_df in annual_state_df.groupby(
    trajectory_group_columns,
    sort=False,
):

    (
        model_name,
        model_code,
        cluster_id,
        unit_id,
    ) = group_values

    unit_df = (
        unit_df
        .sort_values("year")
        .reset_index(drop=True)
    )

    if len(unit_df) < 2:
        continue

    for index in range(
        len(unit_df) - 1
    ):

        first = unit_df.iloc[index]
        second = unit_df.iloc[index + 1]

        year_gap = int(
            second["year"]
            - first["year"]
        )

        if (
            REQUIRE_CONSECUTIVE_YEARS
            and year_gap != 1
        ):
            continue

        observed_transition = (
            cropland_transition_type(
                first["observed_annual_state"],
                second["observed_annual_state"],
            )
        )

        predicted_transition = (
            cropland_transition_type(
                first["predicted_annual_state"],
                second["predicted_annual_state"],
            )
        )

        observed_changed = (
            observed_transition
            in {
                "gain_crop",
                "loss_crop",
            }
        )

        predicted_changed = (
            predicted_transition
            in {
                "gain_crop",
                "loss_crop",
            }
        )

        annual_transition_rows.append({
            "model": model_name,
            "model_code": model_code,
            "cluster_id": int(cluster_id),
            "unit_id": unit_id,

            "year_from": int(
                first["year"]
            ),

            "year_to": int(
                second["year"]
            ),

            "year_gap": year_gap,

            "observed_state_from":
                first["observed_annual_state"],

            "observed_state_to":
                second["observed_annual_state"],

            "predicted_state_from":
                first["predicted_annual_state"],

            "predicted_state_to":
                second["predicted_annual_state"],

            "observed_transition":
                observed_transition,

            "predicted_transition":
                predicted_transition,

            "transition_correct": bool(
                observed_transition
                == predicted_transition
            ),

            "observed_crop_change":
                observed_changed,

            "predicted_crop_change":
                predicted_changed,

            "false_crop_transition": bool(
                predicted_changed
                and not observed_changed
            ),

            "missed_crop_transition": bool(
                observed_changed
                and not predicted_changed
            ),
        })


annual_transition_df = pd.DataFrame(
    annual_transition_rows
)

if annual_transition_df.empty:
    raise RuntimeError(
        "No annual transition pairs were available. "
        "Consider setting REQUIRE_CONSECUTIVE_YEARS=False "
        "if observations contain gaps between years."
    )


# ======================================================================================
# MODEL × CLUSTER ANNUAL TRANSITION METRICS
# ======================================================================================

annual_transition_summary_rows = []

for (
    model_name,
    model_code,
    cluster_id,
), group in annual_transition_df.groupby(
    [
        "model",
        "model_code",
        "cluster_id",
    ],
    sort=True,
):

    truth = (
        group["observed_transition"]
        .to_numpy()
    )

    prediction = (
        group["predicted_transition"]
        .to_numpy()
    )

    gain_metrics = (
        safe_binary_transition_metrics(
            truth,
            prediction,
            "gain_crop",
        )
    )

    loss_metrics = (
        safe_binary_transition_metrics(
            truth,
            prediction,
            "loss_crop",
        )
    )

    stable_crop_metrics = (
        safe_binary_transition_metrics(
            truth,
            prediction,
            "stable_crop",
        )
    )

    stable_noncrop_metrics = (
        safe_binary_transition_metrics(
            truth,
            prediction,
            "stable_non_crop",
        )
    )

    class_f1_values = f1_score(
        truth,
        prediction,
        labels=ANNUAL_TRANSITION_CLASSES,
        average=None,
        zero_division=0,
    )

    class_support = np.array([
        int(
            np.sum(
                truth == transition_class
            )
        )
        for transition_class
        in ANNUAL_TRANSITION_CLASSES
    ])

    supported_transition_f1 = (
        class_f1_values[
            class_support > 0
        ]
    )

    gain_loss_f1_values = []

    if (
        gain_metrics["support"]
        >= MIN_TRANSITION_SUPPORT
    ):
        gain_loss_f1_values.append(
            gain_metrics["f1"]
        )

    if (
        loss_metrics["support"]
        >= MIN_TRANSITION_SUPPORT
    ):
        gain_loss_f1_values.append(
            loss_metrics["f1"]
        )

    crop_gain_loss_macro_f1 = (
        float(
            np.mean(
                gain_loss_f1_values
            )
        )
        if gain_loss_f1_values
        else np.nan
    )

    observed_change = (
        group["observed_crop_change"]
        .to_numpy(dtype=bool)
    )

    predicted_change = (
        group["predicted_crop_change"]
        .to_numpy(dtype=bool)
    )

    annual_transition_summary_rows.append({
        "model": model_name,
        "model_code": model_code,
        "cluster_id": int(cluster_id),

        "n_annual_transitions": int(
            len(group)
        ),

        "n_transition_units": int(
            group["unit_id"].nunique()
        ),

        # Exact four-state transition classification
        "annual_transition_accuracy":
            accuracy_score(
                truth,
                prediction,
            ),

        "annual_transition_macro_f1": (
            float(
                np.mean(
                    supported_transition_f1
                )
            )
            if len(
                supported_transition_f1
            )
            else np.nan
        ),

        # Any gain or loss versus stable
        "crop_change_precision":
            precision_score(
                observed_change,
                predicted_change,
                zero_division=0,
            ),

        "crop_change_recall":
            recall_score(
                observed_change,
                predicted_change,
                zero_division=0,
            ),

        "crop_change_f1":
            f1_score(
                observed_change,
                predicted_change,
                zero_division=0,
            ),

        # Cropland gain
        "gain_support":
            gain_metrics["support"],

        "gain_predicted_n":
            gain_metrics["predicted_n"],

        "gain_precision":
            gain_metrics["precision"],

        "gain_recall":
            gain_metrics["recall"],

        "gain_f1":
            gain_metrics["f1"],

        # Cropland loss
        "loss_support":
            loss_metrics["support"],

        "loss_predicted_n":
            loss_metrics["predicted_n"],

        "loss_precision":
            loss_metrics["precision"],

        "loss_recall":
            loss_metrics["recall"],

        "loss_f1":
            loss_metrics["f1"],

        # Combined gain/loss score
        "crop_gain_loss_macro_f1":
            crop_gain_loss_macro_f1,

        # Stable states
        "stable_crop_support":
            stable_crop_metrics["support"],

        "stable_crop_f1":
            stable_crop_metrics["f1"],

        "stable_noncrop_support":
            stable_noncrop_metrics["support"],

        "stable_noncrop_f1":
            stable_noncrop_metrics["f1"],

        # Error rates
        "false_transition_rate": float(
            group[
                "false_crop_transition"
            ].mean()
        ),

        "missed_transition_rate": float(
            group[
                "missed_crop_transition"
            ].mean()
        ),
    })


annual_transition_summary_df = pd.DataFrame(
    annual_transition_summary_rows
)


# ======================================================================================
# ADD ANNUAL STATE AND POINT-MONTH METRICS
# ======================================================================================

pointmonth_summary = (
    teacher_metrics_df[
        teacher_metrics_df["status"]
        .eq("evaluated")
    ][
        [
            "model",
            "cluster_id",
            "balanced_accuracy",
            "macro_f1",
            "cropland_f1",
        ]
    ]
    .copy()
    .rename(
        columns={
            "balanced_accuracy":
                "pointmonth_balanced_accuracy",
            "macro_f1":
                "pointmonth_macro_f1",
            "cropland_f1":
                "pointmonth_cropland_f1",
        }
    )
)

annual_model_comparison_df = (
    annual_transition_summary_df
    .merge(
        annual_state_summary_df,
        on=[
            "model",
            "model_code",
            "cluster_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        pointmonth_summary,
        on=[
            "model",
            "cluster_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ======================================================================================
# COMPACT MODEL-COMPARISON TABLE
# ======================================================================================

compact_annual_transition_table = (
    annual_model_comparison_df[
        [
            "cluster_id",
            "model_code",

            "n_annual_transitions",
            "gain_support",
            "loss_support",

            "annual_crop_f1",

            "gain_precision",
            "gain_recall",
            "gain_f1",

            "loss_precision",
            "loss_recall",
            "loss_f1",

            "crop_gain_loss_macro_f1",

            "annual_transition_accuracy",
            "annual_transition_macro_f1",

            "false_transition_rate",
            "missed_transition_rate",

            "pointmonth_balanced_accuracy",
            "pointmonth_cropland_f1",
        ]
    ]
    .copy()
)

compact_annual_transition_table.columns = [
    "Cl",
    "Model",

    "Pairs",
    "GainN",
    "LossN",

    "AnnualCropF1",

    "GainPrec",
    "GainRec",
    "GainF1",

    "LossPrec",
    "LossRec",
    "LossF1",

    "GainLossF1",

    "TransitionAcc",
    "TransitionMacroF1",

    "FalseChange",
    "MissedChange",

    "PointBA",
    "PointCropF1",
]

metric_columns = [
    "AnnualCropF1",

    "GainPrec",
    "GainRec",
    "GainF1",

    "LossPrec",
    "LossRec",
    "LossF1",

    "GainLossF1",

    "TransitionAcc",
    "TransitionMacroF1",

    "FalseChange",
    "MissedChange",

    "PointBA",
    "PointCropF1",
]

for column in metric_columns:

    compact_annual_transition_table[column] = (
        compact_annual_transition_table[
            column
        ]
        .round(3)
    )

compact_annual_transition_table = (
    compact_annual_transition_table
    .sort_values(
        [
            "Cl",
            "GainLossF1",
            "GainF1",
            "TransitionMacroF1",
            "FalseChange",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
        ],
        na_position="last",
    )
)


print("\n" + "-" * 120)
print("ANNUAL CROPLAND-TRANSITION PERFORMANCE")
print("-" * 120)

display(
    compact_annual_transition_table
)


# ======================================================================================
# METRIC MATRICES
# ======================================================================================

def show_annual_metric_matrix(
    source,
    value_column,
    title,
    higher_is_better=True,
):

    matrix = (
        source
        .pivot(
            index="model_code",
            columns="cluster_id",
            values=value_column,
        )
        .rename_axis(
            index="Model",
            columns="Cluster",
        )
        .round(3)
    )

    matrix["Mean"] = (
        matrix.mean(axis=1)
        .round(3)
    )

    matrix = matrix.sort_values(
        "Mean",
        ascending=not higher_is_better,
    )

    print("\n" + "-" * 100)
    print(title)
    print("-" * 100)

    display(matrix)

    return matrix


gain_f1_matrix = show_annual_metric_matrix(
    annual_model_comparison_df,
    "gain_f1",
    "ANNUAL CROPLAND-GAIN F1",
    higher_is_better=True,
)

loss_f1_matrix = show_annual_metric_matrix(
    annual_model_comparison_df,
    "loss_f1",
    "ANNUAL CROPLAND-LOSS F1",
    higher_is_better=True,
)

gain_loss_f1_matrix = show_annual_metric_matrix(
    annual_model_comparison_df,
    "crop_gain_loss_macro_f1",
    "ANNUAL CROPLAND GAIN/LOSS MACRO-F1",
    higher_is_better=True,
)

transition_accuracy_matrix = show_annual_metric_matrix(
    annual_model_comparison_df,
    "annual_transition_accuracy",
    "EXACT ANNUAL TRANSITION-TYPE ACCURACY",
    higher_is_better=True,
)

false_change_matrix = show_annual_metric_matrix(
    annual_model_comparison_df,
    "false_transition_rate",
    "FALSE ANNUAL CROPLAND-TRANSITION RATE",
    higher_is_better=False,
)


# ======================================================================================
# SUPPORT WARNING TABLE
# ======================================================================================

low_support_table = (
    annual_model_comparison_df[
        (
            annual_model_comparison_df[
                "gain_support"
            ] < MIN_TRANSITION_SUPPORT
        )
        |
        (
            annual_model_comparison_df[
                "loss_support"
            ] < MIN_TRANSITION_SUPPORT
        )
    ][
        [
            "cluster_id",
            "model_code",
            "gain_support",
            "loss_support",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "cluster_id",
            "model_code",
        ]
    )
)

print("\n" + "-" * 100)
print("GAIN/LOSS SUPPORT CHECK")
print("-" * 100)

if low_support_table.empty:

    print(
        "All model-cluster combinations have sufficient gain and loss support."
    )

else:

    print(
        f"Interpret gain or loss F1 carefully when support is below "
        f"{MIN_TRANSITION_SUPPORT}."
    )

    display(
        low_support_table
    )


# ======================================================================================
# SAVE OUTPUTS
# ======================================================================================

annual_state_df.to_csv(
    TABLE_DIR /
    "annual_observed_predicted_states.csv",
    index=False,
)

annual_transition_df.to_csv(
    TABLE_DIR /
    "annual_cropland_transition_predictions.csv",
    index=False,
)

annual_state_summary_df.to_csv(
    TABLE_DIR /
    "annual_state_metrics_by_model_cluster.csv",
    index=False,
)

annual_transition_summary_df.to_csv(
    TABLE_DIR /
    "annual_transition_metrics_by_model_cluster.csv",
    index=False,
)

annual_model_comparison_df.to_csv(
    TABLE_DIR /
    "annual_transition_model_comparison_full.csv",
    index=False,
)

compact_annual_transition_table.to_csv(
    TABLE_DIR /
    "annual_transition_model_comparison_compact.csv",
    index=False,
)

gain_f1_matrix.to_csv(
    TABLE_DIR /
    "matrix_annual_crop_gain_f1.csv",
)

loss_f1_matrix.to_csv(
    TABLE_DIR /
    "matrix_annual_crop_loss_f1.csv",
)

gain_loss_f1_matrix.to_csv(
    TABLE_DIR /
    "matrix_annual_crop_gain_loss_macro_f1.csv",
)

transition_accuracy_matrix.to_csv(
    TABLE_DIR /
    "matrix_annual_transition_accuracy.csv",
)

false_change_matrix.to_csv(
    TABLE_DIR /
    "matrix_false_annual_transition_rate.csv",
)


# ======================================================================================
# FINAL SUMMARY
# ======================================================================================

print("\n" + "=" * 100)
print("BLOCK 08d SUMMARY")
print("=" * 100)

print(
    "Annual unit states:",
    len(annual_state_df),
)

print(
    "Annual transition pairs:",
    len(annual_transition_df),
)

print(
    "Unique transition units:",
    annual_transition_df[
        "unit_id"
    ].nunique(),
)

print(
    "\nObserved transition counts:"
)

display(
    annual_transition_df[
        [
            "cluster_id",
            "unit_id",
            "year_from",
            "year_to",
            "observed_transition",
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "cluster_id",
            "observed_transition",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print(
    "\nPrimary interpretation:"
)

print(
    "  GainF1 measures annual non-cropland → cropland detection."
)

print(
    "  LossF1 measures annual cropland → non-cropland detection."
)

print(
    "  GainLossF1 combines both when each has sufficient support."
)

print(
    "  TransitionAcc requires the full annual transition type to be correct."
)

print(
    "  FalseChange measures predicted crop gain/loss where none was observed."
)

print(
    "\nThis block is diagnostic only and does not yet alter Block 09 selection."
)


BLOCK 08d — ANNUAL CROPLAND-TRANSITION VALIDATION

------------------------------------------------------------------------------------------------------------------------
ANNUAL CROPLAND-TRANSITION PERFORMANCE
------------------------------------------------------------------------------------------------------------------------


,Cl,Model,Pairs,GainN,LossN,AnnualCropF1,GainPrec,GainRec,GainF1,LossPrec,LossRec,LossF1,GainLossF1,TransitionAcc,TransitionMacroF1,FalseChange,MissedChange,PointBA,PointCropF1
12,1,M03,84,6,4,0.814,0.500,0.667,0.571,0.667,0.500,0.571,0.571,0.857,0.717,0.060,0.048,0.467,0.852
8,1,M02,84,6,4,0.797,0.600,0.500,0.545,0.000,0.000,0.000,0.545,0.821,0.562,0.048,0.083,0.461,0.826
16,1,M04,84,6,4,0.793,0.444,0.667,0.533,1.000,0.500,0.667,0.533,0.869,0.742,0.060,0.048,0.452,0.855
4,1,P04,84,6,4,0.790,0.400,0.333,0.364,0.500,0.500,0.500,0.364,0.821,0.643,0.060,0.071,0.657,0.841
28,1,P02,84,6,4,0.656,0.400,0.333,0.364,0.500,0.750,0.600,0.364,0.774,0.642,0.071,0.060,0.414,0.752
0,1,M01,84,6,4,0.766,0.286,0.333,0.308,0.200,0.250,0.222,0.308,0.762,0.538,0.107,0.083,0.463,0.833
20,1,P03,84,6,4,0.729,0.200,0.167,0.182,0.200,0.250,0.222,0.182,0.738,0.499,0.095,0.095,0.640,0.805
24,1,P01,84,6,4,0.701,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.631,0.355,0.143,0.119,0.637,0.761
9,2,M02,280,22,10,0.810,0.250,0.455,0.323,0.222,0.400,0.286,0.304,0.679,0.516,0.157,0.064,0.641,0.832
17,2,M04,280,22,10,0.794,0.286,0.636,0.394,0.095,0.200,0.129,0.262,0.664,0.492,0.189,0.054,0.656,0.807



----------------------------------------------------------------------------------------------------
ANNUAL CROPLAND-GAIN F1
----------------------------------------------------------------------------------------------------


Cluster,1,2,3,4,Mean
Model,,,,,
M02,0.545,0.323,0.353,0.408,0.407
P04,0.364,0.391,0.462,0.396,0.403
M03,0.571,0.308,0.148,0.353,0.345
M04,0.533,0.394,0.062,0.303,0.323
M01,0.308,0.359,0.364,0.229,0.315
P03,0.182,0.364,0.462,0.211,0.305
P02,0.364,0.203,0.069,0.302,0.234
P01,0.000,0.400,0.222,0.132,0.188



----------------------------------------------------------------------------------------------------
ANNUAL CROPLAND-LOSS F1
----------------------------------------------------------------------------------------------------


...


----------------------------------------------------------------------------------------------------
GAIN/LOSS SUPPORT CHECK
----------------------------------------------------------------------------------------------------
Interpret gain or loss F1 carefully when support is below 5.


,cluster_id,model_code,gain_support,loss_support
0,1,M01,6,4
8,1,M02,6,4
12,1,M03,6,4
16,1,M04,6,4
24,1,P01,6,4
28,1,P02,6,4
20,1,P03,6,4
4,1,P04,6,4
2,3,M01,5,1
10,3,M02,5,1



BLOCK 08d SUMMARY
Annual unit states: 17336
Annual transition pairs: 8128
Unique transition units: 346

Observed transition counts:


observed_transition,gain_crop,loss_crop,stable_crop,stable_non_crop
cluster_id,,,,
1,6,4,20,54
2,22,10,163,85
3,5,1,32,84
4,36,27,99,368



Primary interpretation:
  GainF1 measures annual non-cropland → cropland detection.
  LossF1 measures annual cropland → non-cropland detection.
  GainLossF1 combines both when each has sufficient support.
  TransitionAcc requires the full annual transition type to be correct.
  FalseChange measures predicted crop gain/loss where none was observed.

This block is diagnostic only and does not yet alter Block 09 selection.


## BLOCK 09 — Evaluation synthesis and P04 default decision

This block keeps two results separate:

1. **Diagnostic model rankings** show which candidate performs best for each individual metric.
2. **Operational teacher policy** uses P04 in every cluster for consistency, reproducibility and simpler downstream pseudo-label generation.

The operational P04 choice is deliberate; it is not presented as the winner of every metric.


In [ ]:
# ======================================================================================
# BLOCK 09 — SYNTHESIZE MODEL EVALUATION WITHOUT AUTOMATIC ARCHITECTURE SWITCHING
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 09 — MODEL EVALUATION SYNTHESIS")
print("=" * 100)

evaluated = teacher_metrics_df[
    teacher_metrics_df["status"].eq("evaluated")
].copy()

if evaluated.empty:
    raise RuntimeError("No teacher model could be evaluated.")

evaluated["model_code"] = (
    evaluated["model"]
    .map(MODEL_CODES)
    .fillna(evaluated["model"])
)

# --------------------------------------------------------------------------------------
# A. Best model by point-month metrics
# --------------------------------------------------------------------------------------
pointmonth_best_rows = []

for cluster_id, group in evaluated.groupby("cluster_id", sort=True):
    best_ba = group.loc[group["balanced_accuracy"].idxmax()]
    best_crop = group.loc[group["cropland_f1"].idxmax()]
    best_macro = group.loc[group["macro_f1"].idxmax()]

    pointmonth_best_rows.append({
        "cluster_id": int(cluster_id),
        "best_BA_code": best_ba["model_code"],
        "best_BA": best_ba["balanced_accuracy"],
        "best_macroF1_code": best_macro["model_code"],
        "best_macroF1": best_macro["macro_f1"],
        "best_cropF1_code": best_crop["model_code"],
        "best_cropF1": best_crop["cropland_f1"],
    })

pointmonth_best_df = pd.DataFrame(pointmonth_best_rows)

# --------------------------------------------------------------------------------------
# B. Best model by annual cropland-transition metrics
# --------------------------------------------------------------------------------------
if "annual_model_comparison_df" not in globals():
    raise RuntimeError(
        "annual_model_comparison_df is unavailable. "
        "Run Block 08d before Block 09."
    )

annual_eval = annual_model_comparison_df.copy()
annual_eval["model_code"] = (
    annual_eval["model"]
    .map(MODEL_CODES)
    .fillna(annual_eval.get("model_code", annual_eval["model"]))
)

annual_best_rows = []

for cluster_id, group in annual_eval.groupby("cluster_id", sort=True):
    best_gain = group.loc[group["gain_f1"].idxmax()]
    best_transition = group.loc[group["annual_transition_accuracy"].idxmax()]

    supported = group[
        group["crop_gain_loss_macro_f1"].notna()
    ].copy()

    if supported.empty:
        best_gain_loss_code = "insufficient_support"
        best_gain_loss_f1 = np.nan
    else:
        best_gain_loss = supported.loc[
            supported["crop_gain_loss_macro_f1"].idxmax()
        ]
        best_gain_loss_code = best_gain_loss["model_code"]
        best_gain_loss_f1 = best_gain_loss["crop_gain_loss_macro_f1"]

    annual_best_rows.append({
        "cluster_id": int(cluster_id),
        "best_gainF1_code": best_gain["model_code"],
        "best_gainF1": best_gain["gain_f1"],
        "best_supported_gainloss_code": best_gain_loss_code,
        "best_supported_gainlossF1": best_gain_loss_f1,
        "best_transition_accuracy_code": best_transition["model_code"],
        "best_transition_accuracy": best_transition["annual_transition_accuracy"],
    })

annual_best_df = pd.DataFrame(annual_best_rows)

# --------------------------------------------------------------------------------------
# C. P04 metrics in every cluster
# --------------------------------------------------------------------------------------
p04_model = DEFAULT_OPERATIONAL_TEACHER

p04_point = evaluated[
    evaluated["model"].eq(p04_model)
][
    [
        "cluster_id",
        "model",
        "model_code",
        "n_features",
        "n_train",
        "n_test",
        "balanced_accuracy",
        "macro_f1",
        "cropland_precision",
        "cropland_recall",
        "cropland_f1",
    ]
].copy()

p04_annual = annual_eval[
    annual_eval["model"].eq(p04_model)
][
    [
        "cluster_id",
        "annual_crop_f1",
        "gain_support",
        "gain_precision",
        "gain_recall",
        "gain_f1",
        "loss_support",
        "loss_precision",
        "loss_recall",
        "loss_f1",
        "crop_gain_loss_macro_f1",
        "annual_transition_accuracy",
        "annual_transition_macro_f1",
        "false_transition_rate",
        "missed_transition_rate",
    ]
].copy()

p04_evaluation_df = (
    p04_point
    .merge(
        p04_annual,
        on="cluster_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("cluster_id")
)

evaluation_synthesis_df = (
    pointmonth_best_df
    .merge(
        annual_best_df,
        on="cluster_id",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        p04_evaluation_df[
            [
                "cluster_id",
                "balanced_accuracy",
                "cropland_f1",
                "gain_f1",
                "loss_f1",
                "crop_gain_loss_macro_f1",
                "annual_transition_accuracy",
                "false_transition_rate",
            ]
        ].rename(
            columns={
                "balanced_accuracy": "P04_point_BA",
                "cropland_f1": "P04_point_cropF1",
                "gain_f1": "P04_gainF1",
                "loss_f1": "P04_lossF1",
                "crop_gain_loss_macro_f1": "P04_supported_gainlossF1",
                "annual_transition_accuracy": "P04_transition_accuracy",
                "false_transition_rate": "P04_false_transition_rate",
            }
        ),
        on="cluster_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("cluster_id")
)

for column in evaluation_synthesis_df.columns:
    if pd.api.types.is_float_dtype(evaluation_synthesis_df[column]):
        evaluation_synthesis_df[column] = (
            evaluation_synthesis_df[column].round(3)
        )

print("\nMODEL WINNERS BY INDIVIDUAL DIAGNOSTIC")
display(evaluation_synthesis_df)

print("\nP04 PERFORMANCE BY CLUSTER")
display(
    p04_evaluation_df[
        [
            "cluster_id",
            "n_features",
            "n_train",
            "n_test",
            "balanced_accuracy",
            "cropland_f1",
            "annual_crop_f1",
            "gain_support",
            "gain_f1",
            "loss_support",
            "loss_f1",
            "crop_gain_loss_macro_f1",
            "annual_transition_accuracy",
            "false_transition_rate",
            "missed_transition_rate",
        ]
    ].round(3)
)

print("\nINTERPRETATION")
print(
    "  - Different models win different individual metrics and clusters."
)
print(
    "  - P04 is retained as the common operational teacher because it combines "
    "temporal information and spatial patch context in one consistent RF workflow."
)
print(
    "  - P04 is not assumed to be the statistical winner of every metric."
)

evaluation_synthesis_df.to_csv(
    TABLE_DIR / "model_evaluation_synthesis_by_cluster.csv",
    index=False,
)

p04_evaluation_df.to_csv(
    TABLE_DIR / "P04_evaluation_by_cluster.csv",
    index=False,
)



BLOCK 09 — MODEL EVALUATION SYNTHESIS

MODEL WINNERS BY INDIVIDUAL DIAGNOSTIC


,cluster_id,best_BA_code,best_BA,best_macroF1_code,best_macroF1,best_cropF1_code,best_cropF1,best_gainF1_code,best_gainF1,best_supported_gainloss_code,best_supported_gainlossF1,best_transition_accuracy_code,best_transition_accuracy,P04_point_BA,P04_point_cropF1,P04_gainF1,P04_lossF1,P04_supported_gainlossF1,P04_transition_accuracy,P04_false_transition_rate
0,1,P04,0.657,P04,0.669,M04,0.855,M03,0.571,M03,0.571,M04,0.869,0.657,0.841,0.364,0.500,0.364,0.821,0.060
1,2,P01,0.709,P01,0.565,P01,0.863,P01,0.400,M02,0.304,P01,0.764,0.653,0.858,0.391,0.080,0.236,0.739,0.100
2,3,M02,0.595,M02,0.451,P03,0.758,P04,0.462,P04,0.462,M01,0.795,0.585,0.744,0.462,0.000,0.462,0.787,0.115
3,4,P01,0.721,P01,0.686,P04,0.619,M02,0.408,P04,0.347,M02,0.645,0.686,0.619,0.396,0.298,0.347,0.634,0.175



P04 PERFORMANCE BY CLUSTER


,cluster_id,n_features,n_train,n_test,balanced_accuracy,cropland_f1,annual_crop_f1,gain_support,gain_f1,loss_support,loss_f1,crop_gain_loss_macro_f1,annual_transition_accuracy,false_transition_rate,missed_transition_rate
0,1,798,641,303,0.657,0.841,0.790,6,0.364,4,0.500,0.364,0.821,0.060,0.071
1,2,798,1817,951,0.653,0.858,0.833,22,0.391,10,0.080,0.236,0.739,0.100,0.075
2,3,798,781,353,0.585,0.744,0.738,5,0.462,1,0.000,0.462,0.787,0.115,0.025
3,4,798,3658,1255,0.686,0.619,0.608,36,0.396,27,0.298,0.347,0.634,0.175,0.055



INTERPRETATION
  - Different models win different individual metrics and clusters.
  - P04 is retained as the common operational teacher because it combines temporal information and spatial patch context in one consistent RF workflow.
  - P04 is not assumed to be the statistical winner of every metric.


## BLOCK 10 — Fit and save cluster-specific P04 teachers

One P04 Random Forest is fitted per ecological cluster using all retained snapshot observations from that cluster.

The architecture and feature definition are identical across clusters; only the fitted RF parameters differ because each cluster has its own training observations.


In [ ]:
# ======================================================================================
# BLOCK 10 — FIT AND SAVE CLUSTER-SPECIFIC P04 TEACHERS
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 10 — FIT AND SAVE P04 TEACHERS")
print("=" * 100)

operational_model = DEFAULT_OPERATIONAL_TEACHER
operational_code = MODEL_CODES[operational_model]

if operational_model not in candidate_tables:
    raise RuntimeError(
        "P04 candidate table is unavailable. "
        "Run Blocks 06–08 first and ensure RUN_PATCH_MODELS=True."
    )

if operational_model not in candidate_feature_map:
    raise RuntimeError(
        "P04 feature manifest is unavailable. Run Block 07 first."
    )


# ======================================================================================
# REBUILD P04 OPERATIONAL REGISTRY IF BLOCK 09b WAS NOT RUN
# ======================================================================================

if (
    "operational_teacher_selection_df" not in globals()
    or operational_teacher_selection_df is None
    or operational_teacher_selection_df.empty
):

    print(
        "operational_teacher_selection_df was not found. "
        "Rebuilding the P04 registry directly from Blocks 08 and 08d."
    )

    required_objects = [
        "teacher_metrics_df",
        "annual_model_comparison_df",
    ]

    missing_objects = [
        name
        for name in required_objects
        if name not in globals()
    ]

    if missing_objects:
        raise RuntimeError(
            "Cannot rebuild the P04 registry. Missing objects: "
            f"{missing_objects}. Run Blocks 08 and 08d first."
        )

    operational_rows = []

    cluster_ids = sorted(
        int(value)
        for value in cluster_assignments["cluster_id"]
        .dropna()
        .unique()
    )

    for cluster_id in cluster_ids:

        metric_rows = teacher_metrics_df[
            teacher_metrics_df["cluster_id"]
            .astype(int)
            .eq(cluster_id)
            &
            teacher_metrics_df["model"]
            .eq(operational_model)
            &
            teacher_metrics_df["status"]
            .eq("evaluated")
        ]

        if len(metric_rows) != 1:
            raise RuntimeError(
                f"Expected one evaluated P04 point-month row for "
                f"Cluster {cluster_id}; found {len(metric_rows)}."
            )

        metric_row = metric_rows.iloc[0]

        annual_rows = annual_model_comparison_df[
            annual_model_comparison_df["cluster_id"]
            .astype(int)
            .eq(cluster_id)
            &
            annual_model_comparison_df["model"]
            .eq(operational_model)
        ]

        if len(annual_rows) != 1:
            raise RuntimeError(
                f"Expected one P04 annual-transition row for "
                f"Cluster {cluster_id}; found {len(annual_rows)}."
            )

        annual_row = annual_rows.iloc[0]

        operational_rows.append({
            "cluster_id": cluster_id,

            "operational_model": operational_model,
            "operational_code": operational_code,

            "operational_description": MODEL_DESCRIPTIONS[
                operational_model
            ],

            "selection_policy": "P04_default_all_clusters",

            "selection_reason": (
                "Common RF teacher combining M02 temporal features "
                "with GEE and NICFI patch summaries."
            ),

            "n_features": int(
                metric_row["n_features"]
            ),

            "pointmonth_balanced_accuracy": (
                metric_row["balanced_accuracy"]
            ),

            "pointmonth_macro_f1": (
                metric_row["macro_f1"]
            ),

            "pointmonth_cropland_f1": (
                metric_row["cropland_f1"]
            ),

            "annual_crop_f1": (
                annual_row["annual_crop_f1"]
            ),

            "gain_support": int(
                annual_row["gain_support"]
            ),

            "gain_f1": (
                annual_row["gain_f1"]
            ),

            "loss_support": int(
                annual_row["loss_support"]
            ),

            "loss_f1": (
                annual_row["loss_f1"]
            ),

            "supported_gain_loss_macro_f1": (
                annual_row["crop_gain_loss_macro_f1"]
            ),

            "annual_transition_accuracy": (
                annual_row["annual_transition_accuracy"]
            ),

            "false_transition_rate": (
                annual_row["false_transition_rate"]
            ),

            "missed_transition_rate": (
                annual_row["missed_transition_rate"]
            ),
        })

    operational_teacher_selection_df = (
        pd.DataFrame(
            operational_rows
        )
        .sort_values(
            "cluster_id"
        )
        .reset_index(
            drop=True
        )
    )

    print("\nRebuilt P04 operational registry:")

    display(
        operational_teacher_selection_df[
            [
                "cluster_id",
                "operational_code",
                "n_features",
                "pointmonth_balanced_accuracy",
                "pointmonth_cropland_f1",
                "gain_f1",
                "loss_f1",
                "annual_transition_accuracy",
            ]
        ].round(3)
    )


# ======================================================================================
# P04 TABLE AND FEATURE MANIFEST
# ======================================================================================

table = candidate_tables[
    operational_model
]

feature_columns = candidate_feature_map[
    operational_model
]

print(
    "\nOperational model:",
    operational_code,
    "|",
    operational_model,
)

print(
    "P04 features:",
    len(feature_columns),
)


# ======================================================================================
# FIT ONE CLUSTER-SPECIFIC P04 RANDOM FOREST
# ======================================================================================

final_teacher_registry = []

for _, selected in operational_teacher_selection_df.iterrows():

    cluster_id = int(
        selected["cluster_id"]
    )

    train = table[
        table["cluster_id"].eq(
            cluster_id
        )
        &
        table["label_role"].eq(
            "snapshot"
        )
        &
        ~table["zone_id"].isin(
            discarded_zones
        )
    ].copy()

    if train.empty:
        raise RuntimeError(
            f"No retained P04 snapshot rows available "
            f"for Cluster {cluster_id}."
        )

    missing_features = [
        column
        for column in feature_columns
        if column not in train.columns
    ]

    if missing_features:
        raise RuntimeError(
            f"P04 training table for Cluster {cluster_id} "
            f"is missing {len(missing_features)} required features. "
            f"First missing: {missing_features[:10]}"
        )

    final_model = make_rf()

    sample_weight = compute_sample_weight(
        "balanced",
        train["land_cover_main5"],
    )

    final_model.fit(
        train[feature_columns],
        train["land_cover_main5"],
        rf__sample_weight=sample_weight,
    )

    model_path = (
        MODEL_DIR
        / f"cluster_{cluster_id}__P04__RF_teacher.joblib"
    )

    manifest_path = (
        MODEL_DIR
        / f"cluster_{cluster_id}__P04__features.json"
    )

    joblib.dump(
        final_model,
        model_path,
    )


    # ==================================================================================
    # FEATURE AND MODEL MANIFEST
    # ==================================================================================

    manifest = {
        "cluster_id": cluster_id,

        "model": operational_model,
        "model_code": operational_code,

        "model_description": MODEL_DESCRIPTIONS[
            operational_model
        ],

        "architecture": "RandomForestClassifier",
        "is_cnn": False,

        "selection_policy": selected[
            "selection_policy"
        ],

        "selection_reason": selected[
            "selection_reason"
        ],

        "feature_columns": feature_columns,
        "n_features": len(feature_columns),

        "target_classes": TARGET_CLASSES,

        "discarded_zones": discarded_zones,

        "training_rows": len(train),

        "training_units": int(
            train["unit_id"].nunique()
        ),

        "training_zones": sorted(
            train["zone_id"]
            .unique()
            .tolist()
        ),

        "validation_metrics": {

            "pointmonth_balanced_accuracy": float(
                selected[
                    "pointmonth_balanced_accuracy"
                ]
            ),

            "pointmonth_macro_f1": float(
                selected[
                    "pointmonth_macro_f1"
                ]
            ),

            "pointmonth_cropland_f1": float(
                selected[
                    "pointmonth_cropland_f1"
                ]
            ),

            "annual_crop_f1": float(
                selected[
                    "annual_crop_f1"
                ]
            ),

            "gain_support": int(
                selected[
                    "gain_support"
                ]
            ),

            "gain_f1": float(
                selected[
                    "gain_f1"
                ]
            ),

            "loss_support": int(
                selected[
                    "loss_support"
                ]
            ),

            "loss_f1": float(
                selected[
                    "loss_f1"
                ]
            ),

            "supported_gain_loss_macro_f1": (
                None
                if pd.isna(
                    selected[
                        "supported_gain_loss_macro_f1"
                    ]
                )
                else float(
                    selected[
                        "supported_gain_loss_macro_f1"
                    ]
                )
            ),

            "annual_transition_accuracy": float(
                selected[
                    "annual_transition_accuracy"
                ]
            ),

            "false_transition_rate": float(
                selected[
                    "false_transition_rate"
                ]
            ),

            "missed_transition_rate": float(
                selected[
                    "missed_transition_rate"
                ]
            ),
        },
    }

    with open(
        manifest_path,
        "w",
    ) as file:

        json.dump(
            manifest,
            file,
            indent=2,
        )


    # ==================================================================================
    # REGISTRY ROW
    # ==================================================================================

    final_teacher_registry.append({

        "cluster_id": cluster_id,

        "model": operational_model,
        "model_code": operational_code,

        "model_description": MODEL_DESCRIPTIONS[
            operational_model
        ],

        "architecture": "RandomForestClassifier",
        "is_cnn": False,

        "selection_policy": selected[
            "selection_policy"
        ],

        "model_path": str(
            model_path
        ),

        "feature_manifest_path": str(
            manifest_path
        ),

        "n_features": len(
            feature_columns
        ),

        "n_snapshot_train_rows": len(
            train
        ),

        "n_snapshot_train_units": int(
            train["unit_id"].nunique()
        ),

        "n_snapshot_train_zones": int(
            train["zone_id"].nunique()
        ),

        "trajectory_balanced_accuracy": selected[
            "pointmonth_balanced_accuracy"
        ],

        "trajectory_macro_f1": selected[
            "pointmonth_macro_f1"
        ],

        "trajectory_cropland_f1": selected[
            "pointmonth_cropland_f1"
        ],

        "annual_crop_f1": selected[
            "annual_crop_f1"
        ],

        "gain_support": selected[
            "gain_support"
        ],

        "gain_f1": selected[
            "gain_f1"
        ],

        "loss_support": selected[
            "loss_support"
        ],

        "loss_f1": selected[
            "loss_f1"
        ],

        "annual_transition_accuracy": selected[
            "annual_transition_accuracy"
        ],

        "false_transition_rate": selected[
            "false_transition_rate"
        ],
    })


# ======================================================================================
# SAVE FINAL REGISTRIES
# ======================================================================================

final_teacher_registry_df = (
    pd.DataFrame(
        final_teacher_registry
    )
    .sort_values(
        "cluster_id"
    )
)

final_teacher_registry_df.to_csv(
    TABLE_DIR
    / "final_teacher_registry.csv",
    index=False,
)

# This file is used by Notebook 04.
operational_teacher_selection_df.to_csv(
    TABLE_DIR
    / "operational_teacher_per_cluster.csv",
    index=False,
)


# ======================================================================================
# DISPLAY FINAL TEACHERS
# ======================================================================================

display(
    final_teacher_registry_df[
        [
            "cluster_id",
            "model_code",
            "architecture",
            "n_features",
            "n_snapshot_train_rows",
            "n_snapshot_train_zones",
            "trajectory_balanced_accuracy",
            "trajectory_cropland_f1",
            "gain_support",
            "gain_f1",
            "loss_support",
            "loss_f1",
            "annual_transition_accuracy",
            "false_transition_rate",
        ]
    ].round(3)
)

print(
    "\nFinal P04 teachers saved to:",
    MODEL_DIR,
)


BLOCK 10 — FIT AND SAVE P04 TEACHERS
operational_teacher_selection_df was not found. Rebuilding the P04 registry directly from Blocks 08 and 08d.

Rebuilt P04 operational registry:


,cluster_id,operational_code,n_features,pointmonth_balanced_accuracy,pointmonth_cropland_f1,gain_f1,loss_f1,annual_transition_accuracy
0,1,P04,798,0.657,0.841,0.364,0.500,0.821
1,2,P04,798,0.653,0.858,0.391,0.080,0.739
2,3,P04,798,0.585,0.744,0.462,0.000,0.787
3,4,P04,798,0.686,0.619,0.396,0.298,0.634



Operational model: P04 | M02_PLUS_GEE_NICFI_PATCH_STATS
P04 features: 798


,cluster_id,model_code,architecture,n_features,n_snapshot_train_rows,n_snapshot_train_zones,trajectory_balanced_accuracy,trajectory_cropland_f1,gain_support,gain_f1,loss_support,loss_f1,annual_transition_accuracy,false_transition_rate
0,1,P04,RandomForestClassifier,798,646,4,0.657,0.841,6,0.364,4,0.500,0.821,0.060
1,2,P04,RandomForestClassifier,798,1842,10,0.653,0.858,22,0.391,10,0.080,0.739,0.100
2,3,P04,RandomForestClassifier,798,808,5,0.585,0.744,5,0.462,1,0.000,0.787,0.115
3,4,P04,RandomForestClassifier,798,3717,12,0.686,0.619,36,0.396,27,0.298,0.634,0.175



Final P04 teachers saved to: /content/gdrive/MyDrive/Colab Notebooks/PanAfrica_LU/NICFI_phase02/05_model_runs/03_teacher_model_selection/run_20260718_203109/models


## BLOCK 11 — Final audit and downstream hand-off

The final audit records both the full eight-model evaluation and the operational decision to use cluster-specific P04 Random Forest teachers in downstream pseudo-label generation.


In [ ]:
# ======================================================================================
# BLOCK 11 — FINAL AUDIT SUMMARY
# ======================================================================================
print("\n" + "=" * 100)
print("BLOCK 11 — FINAL AUDIT SUMMARY")
print("=" * 100)

audit = {
    "run_id": RUN_ID,
    "cluster_column": CLUSTER_COLUMN,
    "n_clusters": int(cluster_assignments["cluster_id"].nunique()),
    "n_screened_zones": int(zone_decisions["zone_id"].nunique()),
    "discarded_zones": discarded_zones,
    "n_discarded_zones": len(discarded_zones),
    "n_retained_zones": len(retained_zones),
    "candidate_models": {
        MODEL_CODES[model]: {
            "internal_name": model,
            "description": MODEL_DESCRIPTIONS[model],
            "n_features": len(features),
        }
        for model, features in candidate_feature_map.items()
    },
    "evaluation_layers": {
        "pointmonth": (
            "Manual trajectory observations scored as dated land-cover states."
        ),
        "trajectory_sequence": (
            "Complete unit_id sequences, changing-unit agreement and "
            "excess-transition diagnostics."
        ),
        "annual_cropland_transitions": (
            "Consecutive unit-year pairs classified as stable non-crop, "
            "crop gain, stable crop or crop loss."
        ),
    },
    "operational_policy": {
        "model_code": "P04",
        "model": DEFAULT_OPERATIONAL_TEACHER,
        "description": MODEL_DESCRIPTIONS[DEFAULT_OPERATIONAL_TEACHER],
        "architecture": "RandomForestClassifier",
        "is_cnn": False,
        "same_architecture_all_clusters": True,
        "reason": (
            "Consistent multi-source RF teacher combining temporal features "
            "with GEE and NICFI patch-summary context."
        ),
    },
    "operational_teachers": {
        str(int(row["cluster_id"])): {
            "code": row["model_code"],
            "model": row["model"],
            "model_path": row["model_path"],
            "feature_manifest_path": row["feature_manifest_path"],
            "n_features": int(row["n_features"]),
        }
        for _, row in final_teacher_registry_df.iterrows()
    },
    "training_source": "retained snapshot labels only",
    "validation_source": "retained manual trajectory labels only",
    "exact_unit_overlap_removed": True,
    "bootstrap_unit": "unit_id",
    "n_bootstrap": N_BOOTSTRAP,
    "automatic_zone_exclusion_rule": {
        "cropland_precision_below": AUTO_EXCLUDE_PRECISION_THRESHOLD,
        "cropland_recall_below": AUTO_EXCLUDE_RECALL_THRESHOLD,
        "operator": "AND",
    },
}

with open(RUN_DIR / "run_audit.json", "w") as file:
    json.dump(audit, file, indent=2)

summary_lines = [
    "Teacher-model evaluation and P04 operational registry",
    "=" * 72,
    f"Run: {RUN_ID}",
    f"Clusters: {audit['n_clusters']}",
    f"Zones screened: {audit['n_screened_zones']}",
    f"Discarded zones ({len(discarded_zones)}): {discarded_zones}",
    "",
    "Evaluation completed:",
    "  - Point-month state classification for all 8 candidate RF models.",
    "  - Full trajectory-behaviour diagnostics.",
    "  - Annual cropland gain/loss/stability diagnostics.",
    "",
    "Operational decision:",
    "  - P04 is used in every cluster.",
    "  - P04 is a Random Forest, not a CNN.",
    "  - P04 combines M02 temporal features with GEE and NICFI patch summaries.",
    "",
    "Saved cluster teachers:",
]

for _, row in final_teacher_registry_df.sort_values("cluster_id").iterrows():
    summary_lines.append(
        f"  Cluster {int(row['cluster_id'])}: P04 | "
        f"features={int(row['n_features'])} | "
        f"snapshot rows={int(row['n_snapshot_train_rows'])} | "
        f"point BA={row['trajectory_balanced_accuracy']:.3f} | "
        f"crop F1={row['trajectory_cropland_f1']:.3f} | "
        f"gain F1={row['gain_f1']:.3f} | "
        f"loss F1={row['loss_f1']:.3f}"
    )

summary_lines += [
    "",
    "Validation safeguards:",
    "  - Teachers were trained only on retained snapshot observations.",
    "  - Evaluation used manual trajectory observations.",
    "  - Any trajectory unit_id was removed from snapshot training.",
    "  - Bootstrap uncertainty resampled complete unit_id trajectories.",
    "  - The same audited P04 feature definition is used for validation and final fitting.",
]

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(RUN_DIR / "teacher_selection_summary.txt", "w") as file:
    file.write(summary_text)

print("\nOutputs:")
print("  Evaluation synthesis:", TABLE_DIR / "model_evaluation_synthesis_by_cluster.csv")
print("  P04 evaluation:", TABLE_DIR / "P04_evaluation_by_cluster.csv")
print("  Operational selection:", TABLE_DIR / "operational_teacher_per_cluster.csv")
print("  Final P04 registry:", TABLE_DIR / "final_teacher_registry.csv")
print("  Audit:", RUN_DIR / "run_audit.json")
print("\nNotebook completed.")



BLOCK 11 — FINAL AUDIT SUMMARY
Teacher-model evaluation and P04 operational registry
Run: run_20260718_203109
Clusters: 4
Zones screened: 34
Discarded zones (3): ['G3001_Z146_dynamic_high_change', 'G3002_Z147_dynamic_high_change', 'G3005_Z150_dynamic_high_change']

Evaluation completed:
  - Point-month state classification for all 8 candidate RF models.
  - Full trajectory-behaviour diagnostics.
  - Annual cropland gain/loss/stability diagnostics.

Operational decision:
  - P04 is used in every cluster.
  - P04 is a Random Forest, not a CNN.
  - P04 combines M02 temporal features with GEE and NICFI patch summaries.

Saved cluster teachers:
  Cluster 1: P04 | features=798 | snapshot rows=646 | point BA=0.657 | crop F1=0.841 | gain F1=0.364 | loss F1=0.500
  Cluster 2: P04 | features=798 | snapshot rows=1842 | point BA=0.653 | crop F1=0.858 | gain F1=0.391 | loss F1=0.080
  Cluster 3: P04 | features=798 | snapshot rows=808 | point BA=0.585 | crop F1=0.744 | gain F1=0.462 | loss F1=0.000